# HW4 Deep PromptIR 192-to-224 Fine-Tune + LWR Notebook

This notebook keeps the self-contained Kaggle training path, but is configured for the current best HW4 model family:

- Deep PromptIR large variant: `MODEL_DIM=56`, `NUM_BLOCKS=[4,6,8,12]`, `PROMPT_LEN=12`
- Fine-tune from the best 192-patch checkpoint to 224-patch training
- Optional later LWR fine-tune from the best 224 checkpoint
- Selectable CUDA GPU/DDP or PyTorch/XLA TPU training
- Cosine LR schedule with warmup
- Early stop when validation is flat for 5 epochs
- Save top-k checkpoints and the most recent late checkpoints for ensemble candidates

Default use:

1. Upload or mount the best 192 checkpoint and set `BEST_192_CKPT_PATH`, or let `AUTO_BEST_192` search the configured globs.
2. Run `RUN_ONLY_STAGE = "ft224_from_best192"` with `USE_LOCAL_WEATHER_REFINE = False`.
3. Later, set `RUN_ONLY_STAGE = "ft224_lwr"` and `USE_LOCAL_WEATHER_REFINE = True` to add LWR from the best 224 checkpoint.

For Kaggle TPU, set `ACCELERATOR = "tpu"`. Leave `TPU_NUM_PROCESSES = 1`, `TPU_PROCESS_BOUNDS = "1,1,1"`, and `TPU_VISIBLE_CHIPS = "0"` for the Kaggle-safe single-chip path unless the runtime exposes the full TPU slice correctly.


In [ ]:
import os
import sys
import json
import math
import time
import random
import zipfile
import subprocess
from pathlib import Path

import numpy as np
from PIL import Image

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if torch.cuda.is_available():
    print("CUDA capability:", torch.cuda.get_device_capability(0))

In [ ]:
# =========================
# CAPSLOCKED CONFIG
# =========================

DATA_ROOT = "/kaggle/input/datasets/davonmartono/promptir-image-restoration-hw4"
AUTO_FIND_DATA_ROOT = True

OUTPUT_DIR = "/kaggle/working/output"
EXPERIMENT_NAME = "E032_deep_promptir_ft224_from_best192"
RUN_NAME = EXPERIMENT_NAME
SEED = 42

VAL_PER_TASK = 160
USE_FIXED_SPLIT = True
SPLIT_JSON_PATH = "/kaggle/working/hw4_split_seed42.json"

# Current best architecture family from E028/E029/E030.
MODEL_DIM = 56
NUM_BLOCKS = [4, 6, 8, 12]
NUM_REFINEMENT_BLOCKS = 6
PROMPT_LEN = 12
NUM_HEADS = [1, 2, 4, 8]
FFN_EXPANSION_FACTOR = 2.66
BIAS = False
LAYER_NORM_TYPE = "WithBias"
DUAL_PIXEL_TASK = False
USE_SOFT_PROMPT_ROUTING = False
AUX_CLS_WEIGHT = 0.0

# Global default. Stage dictionaries can override this. For the 224 base stage,
# keep this False. For a later LWR-only run, set RUN_ONLY_STAGE="ft224_lwr"
# and either set this True or rely on the stage override.
USE_LOCAL_WEATHER_REFINE = False
ALLOW_PARTIAL_LOAD_FOR_LWR = True
LWR_EXPANSION = 1.0
LWR_RES_SCALE_INIT = 0.05
LWR_ZERO_INIT_FINAL = True
RUN_LWR_SMOKE_TEST = False

# Checkpoint auto-resolution. Prefer setting BEST_192_CKPT_PATH explicitly to
# the uploaded Kaggle input checkpoint, for example:
# BEST_192_CKPT_PATH = "/kaggle/input/my-hw4-checkpoints/epoch052-psnr29.376.ckpt"
BEST_192_CKPT_PATH = ""
BEST_192_CKPT_GLOBS = [
    "/kaggle/input/**/*192*.ckpt",
    "/kaggle/input/**/epoch052-psnr29.376.ckpt",
    "/kaggle/working/output/runs/**/*192*/*.ckpt",
    "/kaggle/working/output/runs/**/epoch052-psnr29.376.ckpt",
]
BEST_224_CKPT_PATH = ""
BEST_224_CKPT_GLOBS = [
    "/kaggle/input/**/*224*.ckpt",
    "/kaggle/input/**/epoch012-psnr29.433.ckpt",
    "/kaggle/working/output/runs/**/*224*/*.ckpt",
    "/kaggle/working/output/runs/**/epoch012-psnr29.433.ckpt",
]

# Configurable restoration loss. Keep L1 for PSNR-oriented fine-tuning unless
# deliberately running a loss ablation.
LOSS_TYPE = "l1"
USE_MULTI_LOSS = False
USE_MS_SSIM = True
L1_WEIGHT = 1.0
SSIM_WEIGHT = 0.1
CHARBONNIER_WEIGHT = 0.02
GRADIENT_WEIGHT = 0.01
CHARBONNIER_EPS = 1e-3
MS_SSIM_LEVELS = 4
SSIM_WINDOW_SIZE = 7

# Accelerator options: "gpu", "tpu", or "auto".
ACCELERATOR = "gpu"
USE_DDP = True
NUM_GPUS = "AUTO"
TPU_NUM_PROCESSES = 1
TPU_PROCESS_BOUNDS = "1,1,1"
TPU_VISIBLE_CHIPS = "0"
TPU_USE_BF16 = True
PRECISION = "AMP"
MATMUL_PRECISION = "medium"

TRAIN_MODE = "MULTI_STAGE"
RESUME_CHECKPOINT = None

# Base 224 fine-tune defaults.
PATCH_SIZE = 224
EPOCHS = 30
BATCH_SIZE = 1
LR = 1e-5
WEIGHT_DECAY = 1e-4
WARMUP_EPOCHS = 2
LR_SCHEDULER = "cosine"
COSINE_ETA_MIN = 0.0
GRAD_CLIP_NORM = 0.75

# Early stop when validation stays flat. A change <= EARLY_STOP_MIN_DELTA dB
# counts as flat. Set patience to 0 or None to disable.
EARLY_STOP_FLAT_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 1e-3

# Checkpoint retention for ensemble selection.
SAVE_TOP_K = 5
SAVE_RECENT_K = 5
SAVE_LAST = True

FT224_STAGE_NAME = "ft224_from_best192"
LWR_STAGE_NAME = "ft224_lwr"

STAGES = [
    {
        "NAME": FT224_STAGE_NAME,
        "INIT_CKPT": "AUTO_BEST_192",
        "PATCH_SIZE": 224,
        "EPOCHS": 30,
        "BATCH_SIZE_PER_GPU": 1,
        "LR": 1e-5,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "WARMUP_EPOCHS": 2,
        "LR_SCHEDULER": LR_SCHEDULER,
        "COSINE_ETA_MIN": COSINE_ETA_MIN,
        "GRAD_CLIP": 0.75,
        "USE_LOCAL_WEATHER_REFINE": False,
        "ALLOW_PARTIAL_LOAD_FOR_LWR": False,
        "EARLY_STOP_FLAT_PATIENCE": EARLY_STOP_FLAT_PATIENCE,
        "EARLY_STOP_MIN_DELTA": EARLY_STOP_MIN_DELTA,
    },
    {
        "NAME": LWR_STAGE_NAME,
        # LWR is usually launched in a fresh Kaggle run from an uploaded
        # best 224 checkpoint. Set BEST_224_CKPT_PATH explicitly when possible.
        "INIT_CKPT": "AUTO_BEST_224",
        "PATCH_SIZE": 224,
        "EPOCHS": 10,
        "BATCH_SIZE_PER_GPU": 1,
        "LR": 5e-6,
        "WEIGHT_DECAY": WEIGHT_DECAY,
        "WARMUP_EPOCHS": 1,
        "LR_SCHEDULER": LR_SCHEDULER,
        "COSINE_ETA_MIN": COSINE_ETA_MIN,
        "GRAD_CLIP": 0.75,
        "USE_LOCAL_WEATHER_REFINE": True,
        "ALLOW_PARTIAL_LOAD_FOR_LWR": True,
        "LWR_EXPANSION": 1.0,
        "LWR_RES_SCALE_INIT": 0.05,
        "LWR_ZERO_INIT_FINAL": True,
        "EARLY_STOP_FLAT_PATIENCE": EARLY_STOP_FLAT_PATIENCE,
        "EARLY_STOP_MIN_DELTA": EARLY_STOP_MIN_DELTA,
    },
]

# First run the 224 fine-tune. Later set RUN_ONLY_STAGE = LWR_STAGE_NAME to add LWR.
# Set RUN_ONLY_STAGE = None to run all configured stages sequentially.
RUN_ONLY_STAGE = FT224_STAGE_NAME

STAGE_NAMES = [stage["NAME"] for stage in STAGES]
if RUN_ONLY_STAGE is not None and RUN_ONLY_STAGE not in STAGE_NAMES:
    print(f"WARNING: RUN_ONLY_STAGE={RUN_ONLY_STAGE!r} is not in STAGES={STAGE_NAMES}. Falling back to the first configured stage.")
    RUN_ONLY_STAGE = STAGE_NAMES[0]

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
TRAIN_LOG_EVERY_N_STEPS = 0
SHOW_PROGRESS_BARS = False

VALIDATE_EVERY_EPOCH = True

USE_TILED_VALIDATION = True
TILE_SIZE = 256
TILE_OVERLAP = 32
USE_X8_TTA = True
INFERENCE_STAGE = None

INFERENCE_CKPT = "AUTO_BEST"
# Expensive post-training actions are opt-in so long Kaggle runs do not time out
# after training has already produced useful checkpoints. Run validation/submission
# in a separate shorter session when needed.
RUN_FINAL_VALIDATION = False
MAKE_SUBMISSION = False
SUBMISSION_NPZ_PATH = "/kaggle/working/pred.npz"
SUBMISSION_ZIP_PATH = "/kaggle/working/submission.zip"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
torch.set_float32_matmul_precision(MATMUL_PRECISION)
print("Configured experiment:", EXPERIMENT_NAME)
print("Configured RUN_NAME:", RUN_NAME)
print("Output dir:", OUTPUT_DIR)
print("Accelerator config:", {"accelerator": ACCELERATOR, "use_ddp": USE_DDP, "num_gpus": NUM_GPUS, "tpu_num_processes": TPU_NUM_PROCESSES, "tpu_process_bounds": TPU_PROCESS_BOUNDS, "tpu_visible_chips": TPU_VISIBLE_CHIPS, "tpu_use_bf16": TPU_USE_BF16})
print("RUN_ONLY_STAGE:", RUN_ONLY_STAGE)
print("Stage list:", [stage["NAME"] for stage in STAGES])
print("Best 192 checkpoint path:", BEST_192_CKPT_PATH or "AUTO_BEST_192 via BEST_192_CKPT_GLOBS")
print("Checkpoint retention:", {"save_top_k": SAVE_TOP_K, "save_recent_k": SAVE_RECENT_K, "save_last": SAVE_LAST})
print("Early stop:", {"patience": EARLY_STOP_FLAT_PATIENCE, "min_delta": EARLY_STOP_MIN_DELTA})
print("LWR default config:", {
    "enabled": USE_LOCAL_WEATHER_REFINE,
    "allow_partial_load": ALLOW_PARTIAL_LOAD_FOR_LWR,
    "expansion": LWR_EXPANSION,
    "residual_scale_init": LWR_RES_SCALE_INIT,
    "zero_init_final": LWR_ZERO_INIT_FINAL,
})
print("LR scheduler config:", {
    "scheduler": LR_SCHEDULER,
    "warmup_epochs": WARMUP_EPOCHS,
    "cosine_eta_min": COSINE_ETA_MIN,
})
print("Post-train actions:", {"run_final_validation": RUN_FINAL_VALIDATION, "make_submission": MAKE_SUBMISSION})
print("Loss config:", {
    "loss_type": LOSS_TYPE,
    "use_multi_loss": USE_MULTI_LOSS,
    "use_ms_ssim": USE_MS_SSIM,
    "l1": L1_WEIGHT,
    "ssim": SSIM_WEIGHT,
    "charbonnier": CHARBONNIER_WEIGHT,
    "gradient": GRADIENT_WEIGHT,
})


In [ ]:
def _has_hw4_layout(path):
    path = Path(path)
    return (
        (path / "train" / "degraded").is_dir()
        and (path / "train" / "clean").is_dir()
        and (path / "test" / "degraded").is_dir()
    )


def find_hw4_root(preferred, auto=True):
    preferred = Path(preferred)
    if preferred.exists() and _has_hw4_layout(preferred):
        return preferred
    if not auto:
        raise FileNotFoundError(f"DATA_ROOT does not match expected HW4 layout: {preferred}")
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        root_path = Path(root)
        if _has_hw4_layout(root_path):
            candidates.append(root_path)
    if not candidates:
        raise FileNotFoundError("Could not auto-find a folder with train/degraded, train/clean, test/degraded under /kaggle/input")
    candidates = sorted(candidates, key=lambda p: (len(str(p)), str(p)))
    return candidates[0]


def _parse_task_index(path):
    stem = Path(path).stem
    task, idx = stem.split("-", 1)
    return task, int(idx)


DATA_ROOT = str(find_hw4_root(DATA_ROOT, AUTO_FIND_DATA_ROOT))
print("Selected DATA_ROOT:", DATA_ROOT)

TRAIN_DEGRADED_DIR = Path(DATA_ROOT) / "train" / "degraded"
TRAIN_CLEAN_DIR = Path(DATA_ROOT) / "train" / "clean"
TEST_DEGRADED_DIR = Path(DATA_ROOT) / "test" / "degraded"

for required in [TRAIN_DEGRADED_DIR, TRAIN_CLEAN_DIR, TEST_DEGRADED_DIR]:
    assert required.is_dir(), f"Missing required folder: {required}"

rain_files = sorted(TRAIN_DEGRADED_DIR.glob("rain-*.png"), key=lambda p: _parse_task_index(p)[1])
snow_files = sorted(TRAIN_DEGRADED_DIR.glob("snow-*.png"), key=lambda p: _parse_task_index(p)[1])
test_files = sorted(TEST_DEGRADED_DIR.glob("*.png"), key=lambda p: int(p.stem) if p.stem.isdigit() else p.name)

print("Rain degraded:", len(rain_files))
print("Snow degraded:", len(snow_files))
print("Test degraded:", len(test_files))

assert rain_files, "No rain training images found"
assert snow_files, "No snow training images found"
assert test_files, "No test images found"

for p in rain_files + snow_files:
    task, idx = _parse_task_index(p)
    clean = TRAIN_CLEAN_DIR / f"{task}_clean-{idx}.png"
    assert clean.exists(), f"Missing clean pair for {p.name}: expected {clean.name}"

print("Pairing check passed.")
print("First test keys:", [p.name for p in test_files[:5]])

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def write_config_py(path="/kaggle/working/hw4_config.py"):
    config_names = [
        "DATA_ROOT", "AUTO_FIND_DATA_ROOT", "OUTPUT_DIR", "EXPERIMENT_NAME",
        "RUN_NAME", "SEED", "VAL_PER_TASK", "USE_FIXED_SPLIT",
        "SPLIT_JSON_PATH", "MODEL_DIM", "NUM_BLOCKS",
        "NUM_REFINEMENT_BLOCKS", "PROMPT_LEN", "NUM_HEADS",
        "FFN_EXPANSION_FACTOR", "BIAS", "LAYER_NORM_TYPE",
        "DUAL_PIXEL_TASK", "USE_SOFT_PROMPT_ROUTING", "AUX_CLS_WEIGHT",
        "USE_LOCAL_WEATHER_REFINE", "ALLOW_PARTIAL_LOAD_FOR_LWR",
        "LWR_EXPANSION", "LWR_RES_SCALE_INIT", "LWR_ZERO_INIT_FINAL",
        "RUN_LWR_SMOKE_TEST", "BEST_192_CKPT_PATH", "BEST_192_CKPT_GLOBS",
        "BEST_224_CKPT_PATH", "BEST_224_CKPT_GLOBS",
        "LOSS_TYPE", "USE_MULTI_LOSS", "USE_MS_SSIM",
        "L1_WEIGHT", "SSIM_WEIGHT", "CHARBONNIER_WEIGHT",
        "GRADIENT_WEIGHT", "CHARBONNIER_EPS", "MS_SSIM_LEVELS",
        "SSIM_WINDOW_SIZE", "ACCELERATOR", "USE_DDP", "NUM_GPUS",
        "TPU_NUM_PROCESSES", "TPU_PROCESS_BOUNDS", "TPU_VISIBLE_CHIPS",
        "TPU_USE_BF16", "PRECISION", "MATMUL_PRECISION", "TRAIN_MODE",
        "RESUME_CHECKPOINT", "STAGES", "RUN_ONLY_STAGE", "FT224_STAGE_NAME",
        "LWR_STAGE_NAME", "PATCH_SIZE", "EPOCHS", "BATCH_SIZE", "LR",
        "WEIGHT_DECAY", "WARMUP_EPOCHS", "LR_SCHEDULER",
        "COSINE_ETA_MIN", "GRAD_CLIP_NORM", "EARLY_STOP_FLAT_PATIENCE",
        "EARLY_STOP_MIN_DELTA", "NUM_WORKERS", "PIN_MEMORY", "PERSISTENT_WORKERS",
        "TRAIN_LOG_EVERY_N_STEPS", "SHOW_PROGRESS_BARS",
        "VALIDATE_EVERY_EPOCH", "SAVE_TOP_K", "SAVE_RECENT_K", "SAVE_LAST",
        "USE_TILED_VALIDATION", "TILE_SIZE", "TILE_OVERLAP", "USE_X8_TTA",
        "INFERENCE_STAGE", "INFERENCE_CKPT", "RUN_FINAL_VALIDATION",
        "MAKE_SUBMISSION", "SUBMISSION_NPZ_PATH", "SUBMISSION_ZIP_PATH",
    ]
    lines = ["# Auto-generated by the Kaggle notebook. Do not edit by hand.\n"]
    for name in config_names:
        lines.append(f"{name} = {repr(globals()[name])}\n")
    Path(path).write_text("".join(lines), encoding="utf-8")
    print("Wrote", path)


seed_everything(SEED)
write_config_py()


In [ ]:
# The actual Dataset/DataLoader classes are written into /kaggle/working/hw4_lib.py in the DDP script generation cell.
# This lightweight split preview mirrors the training split logic.
def make_split_preview():
    rng = random.Random(SEED)
    split_path = Path(SPLIT_JSON_PATH)
    if USE_FIXED_SPLIT and split_path.exists():
        split = json.loads(split_path.read_text())
        print("Loaded existing split:", split_path)
        return split
    rain_names = [p.name for p in rain_files]
    snow_names = [p.name for p in snow_files]
    rng.shuffle(rain_names)
    rng.shuffle(snow_names)
    split = {
        "seed": SEED,
        "val_per_task": VAL_PER_TASK,
        "val": {"rain": sorted(rain_names[:VAL_PER_TASK]), "snow": sorted(snow_names[:VAL_PER_TASK])},
        "train": {"rain": sorted(rain_names[VAL_PER_TASK:]), "snow": sorted(snow_names[VAL_PER_TASK:])},
    }
    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_path.write_text(json.dumps(split, indent=2), encoding="utf-8")
    print("Saved split:", split_path)
    return split


SPLIT = make_split_preview()
print("Train rain/snow:", len(SPLIT["train"]["rain"]), len(SPLIT["train"]["snow"]))
print("Val rain/snow:", len(SPLIT["val"]["rain"]), len(SPLIT["val"]["snow"]))

In [ ]:
# Model implementation summary:
# - PromptIR/Restormer-style encoder-decoder
# - Dynamic channels derived from MODEL_DIM
# - Prompt blocks with PROMPT_LEN learnable components
# - Residual output: restored = network_output + input
#
# The complete implementation is written into /kaggle/working/hw4_lib.py in the next cell.
print("Model config:")
print({
    "MODEL_DIM": MODEL_DIM,
    "NUM_BLOCKS": NUM_BLOCKS,
    "NUM_REFINEMENT_BLOCKS": NUM_REFINEMENT_BLOCKS,
    "PROMPT_LEN": PROMPT_LEN,
    "NUM_HEADS": NUM_HEADS,
    "USE_SOFT_PROMPT_ROUTING": USE_SOFT_PROMPT_ROUTING,
})

In [ ]:
LIB_CODE = r'''
import os
import csv
import json
import glob
import math
import time
import random
import shutil
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


xm = None
xmp = None
xla_pl = None
XLA_IMPORT_ERROR = None


def require_xla():
    global xm, xmp, xla_pl, XLA_IMPORT_ERROR
    if xm is not None and xmp is not None and xla_pl is not None:
        return xm, xmp, xla_pl
    try:
        import torch_xla.core.xla_model as _xm
        import torch_xla.distributed.xla_multiprocessing as _xmp
        import torch_xla.distributed.parallel_loader as _xla_pl
    except Exception as exc:
        XLA_IMPORT_ERROR = exc
        raise RuntimeError(
            "ACCELERATOR='tpu' requires torch_xla. Use a Kaggle TPU runtime "
            "or install a torch_xla build matching the notebook PyTorch version."
        ) from exc
    xm, xmp, xla_pl = _xm, _xmp, _xla_pl
    return xm, xmp, xla_pl


try:
    from pytorch_msssim import ms_ssim as _pt_ms_ssim
    from pytorch_msssim import ssim as _pt_ssim
    HAS_PYTORCH_MSSSIM = True
except Exception:
    _pt_ms_ssim = None
    _pt_ssim = None
    HAS_PYTORCH_MSSSIM = False


def gaussian_window(window_size, sigma, channels, device, dtype):
    coords = torch.arange(window_size, device=device, dtype=dtype) - window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = torch.outer(g, g)
    return window_2d.view(1, 1, window_size, window_size).repeat(channels, 1, 1, 1)


def ssim_components(x, y, window_size=7, sigma=1.5, data_range=1.0):
    x = x.float().clamp(0.0, data_range)
    y = y.float().clamp(0.0, data_range)
    channels = x.shape[1]
    window = gaussian_window(window_size, sigma, channels, x.device, x.dtype)
    padding = window_size // 2
    mu_x = F.conv2d(x, window, padding=padding, groups=channels)
    mu_y = F.conv2d(y, window, padding=padding, groups=channels)
    mu_x2 = mu_x.pow(2)
    mu_y2 = mu_y.pow(2)
    mu_xy = mu_x * mu_y

    sigma_x2 = F.conv2d(x * x, window, padding=padding, groups=channels) - mu_x2
    sigma_y2 = F.conv2d(y * y, window, padding=padding, groups=channels) - mu_y2
    sigma_xy = F.conv2d(x * y, window, padding=padding, groups=channels) - mu_xy

    c1 = (0.01 * data_range) ** 2
    c2 = (0.03 * data_range) ** 2
    cs = ((2.0 * sigma_xy + c2) / (sigma_x2 + sigma_y2 + c2)).clamp(0.0, 1.0)
    ssim = (((2.0 * mu_xy + c1) / (mu_x2 + mu_y2 + c1)) * cs).clamp(0.0, 1.0)
    return ssim.flatten(1).mean(dim=1), cs.flatten(1).mean(dim=1)


def local_ssim_value(x, y, window_size=7):
    ssim, _ = ssim_components(x, y, window_size=window_size)
    return ssim.mean()


def ms_ssim_weights(levels, device=None, dtype=None):
    levels = int(levels)
    base = torch.tensor([0.0448, 0.2856, 0.3001, 0.2363, 0.1333], device=device, dtype=dtype)
    if levels < 1 or levels > len(base):
        raise ValueError(f"MS_SSIM_LEVELS must be in [1, {len(base)}], got {levels}")
    weights = base[:levels]
    return weights / weights.sum()


def local_ms_ssim_value(x, y, levels=4, window_size=7):
    weights = ms_ssim_weights(levels, x.device, x.dtype)
    mcs = []
    for level in range(levels):
        ssim, cs = ssim_components(x, y, window_size=window_size)
        if level < levels - 1:
            mcs.append(cs)
            x = F.avg_pool2d(x, kernel_size=2, stride=2)
            y = F.avg_pool2d(y, kernel_size=2, stride=2)
    value = torch.ones_like(ssim)
    for cs_value, weight in zip(mcs, weights[:-1]):
        value = value * cs_value.clamp_min(1e-6).pow(weight)
    value = value * ssim.clamp_min(1e-6).pow(weights[-1])
    return value.mean()


def structural_similarity_loss(pred, target, use_ms_ssim=True, levels=4, window_size=7):
    pred_for_ssim = pred.float().clamp(0.0, 1.0)
    target_for_ssim = target.float().clamp(0.0, 1.0)
    if HAS_PYTORCH_MSSSIM:
        if use_ms_ssim:
            weights = ms_ssim_weights(levels).tolist()
            value = _pt_ms_ssim(
                pred_for_ssim,
                target_for_ssim,
                data_range=1.0,
                size_average=True,
                win_size=window_size,
                weights=weights,
            )
        else:
            value = _pt_ssim(
                pred_for_ssim,
                target_for_ssim,
                data_range=1.0,
                size_average=True,
                win_size=window_size,
            )
    elif use_ms_ssim:
        value = local_ms_ssim_value(pred_for_ssim, target_for_ssim, levels=levels, window_size=window_size)
    else:
        value = local_ssim_value(pred_for_ssim, target_for_ssim, window_size=window_size)
    return 1.0 - value.clamp(0.0, 1.0)


def charbonnier_loss(pred, target, eps=1e-3):
    return torch.sqrt((pred - target).pow(2) + eps * eps).mean()


def sobel_gradient_loss(pred, target):
    pred = pred.float()
    target = target.float()
    channels = pred.shape[1]
    kx = pred.new_tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3) / 8.0
    ky = pred.new_tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).view(1, 1, 3, 3) / 8.0
    kx = kx.repeat(channels, 1, 1, 1)
    ky = ky.repeat(channels, 1, 1, 1)
    pred_dx = F.conv2d(pred, kx, padding=1, groups=channels)
    pred_dy = F.conv2d(pred, ky, padding=1, groups=channels)
    target_dx = F.conv2d(target, kx, padding=1, groups=channels)
    target_dy = F.conv2d(target, ky, padding=1, groups=channels)
    return F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)


def detached_loss_dict(total, l1, ssim_loss, charb, grad):
    return {
        "loss_l1": l1.detach(),
        "loss_ssim": ssim_loss.detach(),
        "loss_charbonnier": charb.detach(),
        "loss_gradient": grad.detach(),
        "loss_total": total.detach(),
    }


class RestorationMultiLoss(nn.Module):
    def __init__(
        self,
        use_ms_ssim=True,
        l1_weight=1.0,
        ssim_weight=0.2,
        charbonnier_weight=0.05,
        gradient_weight=0.03,
        charbonnier_eps=1e-3,
        ms_ssim_levels=4,
        ssim_window_size=7,
    ):
        super().__init__()
        self.use_ms_ssim = use_ms_ssim
        self.l1_weight = l1_weight
        self.ssim_weight = ssim_weight
        self.charbonnier_weight = charbonnier_weight
        self.gradient_weight = gradient_weight
        self.charbonnier_eps = charbonnier_eps
        self.ms_ssim_levels = ms_ssim_levels
        self.ssim_window_size = ssim_window_size

    def forward(self, pred, target):
        pred_f = pred.float()
        target_f = target.float()
        l1 = F.l1_loss(pred_f, target_f)
        ssim_loss = structural_similarity_loss(
            pred_f,
            target_f,
            use_ms_ssim=self.use_ms_ssim,
            levels=self.ms_ssim_levels,
            window_size=self.ssim_window_size,
        )
        charb = charbonnier_loss(pred_f, target_f, eps=self.charbonnier_eps)
        grad = sobel_gradient_loss(pred_f, target_f)
        total = (
            self.l1_weight * l1
            + self.ssim_weight * ssim_loss
            + self.charbonnier_weight * charb
            + self.gradient_weight * grad
        )
        return total, detached_loss_dict(total, l1, ssim_loss, charb, grad)


class RestorationL1Loss(nn.Module):
    def forward(self, pred, target):
        l1 = F.l1_loss(pred.float(), target.float())
        zero = l1.detach().new_zeros(())
        return l1, detached_loss_dict(l1, l1, zero, zero, zero)


def normalized_loss_type(cfg):
    loss_type = getattr(cfg, "LOSS_TYPE", None)
    if loss_type is None:
        return "multi" if getattr(cfg, "USE_MULTI_LOSS", False) else "l1"
    return str(loss_type).strip().lower().replace("-", "_")


def build_restoration_loss(cfg):
    loss_type = normalized_loss_type(cfg)
    if loss_type in {"l1", "mae"}:
        return RestorationL1Loss()
    if loss_type in {"multi", "multi_loss", "restoration_multi"} or getattr(cfg, "USE_MULTI_LOSS", False):
        return RestorationMultiLoss(
            use_ms_ssim=cfg.USE_MS_SSIM,
            l1_weight=cfg.L1_WEIGHT,
            ssim_weight=cfg.SSIM_WEIGHT,
            charbonnier_weight=cfg.CHARBONNIER_WEIGHT,
            gradient_weight=cfg.GRADIENT_WEIGHT,
            charbonnier_eps=cfg.CHARBONNIER_EPS,
            ms_ssim_levels=cfg.MS_SSIM_LEVELS,
            ssim_window_size=cfg.SSIM_WINDOW_SIZE,
        )
    raise ValueError(f"Unsupported LOSS_TYPE: {getattr(cfg, 'LOSS_TYPE', None)}")


def restoration_loss_description(cfg):
    loss_type = normalized_loss_type(cfg)
    if loss_type in {"l1", "mae"}:
        return "plain L1"
    flavor = "MS-SSIM" if cfg.USE_MS_SSIM else "SSIM"
    return (
        f"L1*{cfg.L1_WEIGHT:g} + {flavor}*{cfg.SSIM_WEIGHT:g} + "
        f"Charbonnier*{cfg.CHARBONNIER_WEIGHT:g} + Gradient*{cfg.GRADIENT_WEIGHT:g}"
    )


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def accelerator_name(cfg):
    name = str(getattr(cfg, "ACCELERATOR", "gpu")).strip().lower()
    if name == "auto":
        if os.environ.get("PJRT_DEVICE", "").upper() == "TPU" or os.environ.get("TPU_NAME"):
            return "tpu"
        return "gpu" if torch.cuda.is_available() else "cpu"
    if name in {"xla", "tpu-vm"}:
        return "tpu"
    return name


def wants_tpu(cfg):
    return accelerator_name(cfg) == "tpu"


def _maybe_set_env(name, value):
    if value is None:
        os.environ.pop(name, None)
        return
    if isinstance(value, str) and value.strip().lower() in {"", "none", "null"}:
        os.environ.pop(name, None)
        return
    os.environ[name] = str(value)


def configure_tpu_environment(cfg):
    if not wants_tpu(cfg):
        return
    os.environ.setdefault("PJRT_DEVICE", "TPU")
    _maybe_set_env("TPU_PROCESS_BOUNDS", getattr(cfg, "TPU_PROCESS_BOUNDS", None))
    _maybe_set_env("TPU_VISIBLE_CHIPS", getattr(cfg, "TPU_VISIBLE_CHIPS", None))
    if bool(getattr(cfg, "TPU_USE_BF16", False)):
        os.environ.setdefault("XLA_USE_BF16", "1")


def _safe_single_chip_tpu(cfg):
    bounds = getattr(cfg, "TPU_PROCESS_BOUNDS", None)
    visible = getattr(cfg, "TPU_VISIBLE_CHIPS", None)
    return str(bounds).strip() == "1,1,1" and str(visible).strip() in {"0", "0,"}


def resolve_tpu_nprocs(cfg):
    value = getattr(cfg, "TPU_NUM_PROCESSES", "AUTO")
    if isinstance(value, str) and value.strip().lower() in {"", "auto", "none", "null"}:
        return 1 if _safe_single_chip_tpu(cfg) else None
    return int(value)


def select_single_process_device(cfg):
    if wants_tpu(cfg):
        configure_tpu_environment(cfg)
        xm_mod, _, _ = require_xla()
        return xm_mod.xla_device()
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    return torch.device("cpu")


def checkpoint_map_location_for_device(device):
    return "cpu" if str(device).startswith("xla") else device


def is_rank0():
    if xm is not None:
        try:
            return bool(xm.is_master_ordinal())
        except Exception:
            pass
    return int(os.environ.get("RANK", "0")) == 0


def log_rank0(*args, **kwargs):
    if is_rank0():
        print(*args, **kwargs, flush=True)


def has_hw4_layout(path):
    path = Path(path)
    return (
        (path / "train" / "degraded").is_dir()
        and (path / "train" / "clean").is_dir()
        and (path / "test" / "degraded").is_dir()
    )


def find_hw4_root(preferred, auto=True):
    preferred = Path(preferred)
    if preferred.exists() and has_hw4_layout(preferred):
        return preferred
    if not auto:
        raise FileNotFoundError(f"DATA_ROOT does not match expected HW4 layout: {preferred}")
    candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        root_path = Path(root)
        if has_hw4_layout(root_path):
            candidates.append(root_path)
    if not candidates:
        raise FileNotFoundError("Could not auto-find HW4 dataset under /kaggle/input")
    return sorted(candidates, key=lambda p: (len(str(p)), str(p)))[0]


def parse_task_index(filename):
    stem = Path(filename).stem
    task, idx = stem.split("-", 1)
    return task, int(idx)


def sorted_image_files(path):
    path = Path(path)
    files = list(path.glob("*.png"))
    def key_fn(p):
        if p.stem.isdigit():
            return (0, int(p.stem))
        task, idx = parse_task_index(p.name)
        return (1 if task == "rain" else 2, idx)
    return sorted(files, key=key_fn)


def build_or_load_split(cfg):
    root = find_hw4_root(cfg.DATA_ROOT, cfg.AUTO_FIND_DATA_ROOT)
    split_path = Path(cfg.SPLIT_JSON_PATH)
    if cfg.USE_FIXED_SPLIT and split_path.exists():
        split = json.loads(split_path.read_text())
        return root, split

    degraded_dir = root / "train" / "degraded"
    clean_dir = root / "train" / "clean"
    rain = sorted(degraded_dir.glob("rain-*.png"), key=lambda p: parse_task_index(p.name)[1])
    snow = sorted(degraded_dir.glob("snow-*.png"), key=lambda p: parse_task_index(p.name)[1])
    for p in rain + snow:
        task, idx = parse_task_index(p.name)
        clean = clean_dir / f"{task}_clean-{idx}.png"
        if not clean.exists():
            raise FileNotFoundError(f"Missing clean pair for {p.name}: {clean}")
    rng = random.Random(cfg.SEED)
    rain_names = [p.name for p in rain]
    snow_names = [p.name for p in snow]
    rng.shuffle(rain_names)
    rng.shuffle(snow_names)
    split = {
        "seed": cfg.SEED,
        "val_per_task": cfg.VAL_PER_TASK,
        "val": {
            "rain": sorted(rain_names[: cfg.VAL_PER_TASK]),
            "snow": sorted(snow_names[: cfg.VAL_PER_TASK]),
        },
        "train": {
            "rain": sorted(rain_names[cfg.VAL_PER_TASK :]),
            "snow": sorted(snow_names[cfg.VAL_PER_TASK :]),
        },
    }
    split_path.parent.mkdir(parents=True, exist_ok=True)
    split_path.write_text(json.dumps(split, indent=2), encoding="utf-8")
    return root, split


def pil_to_tensor(path):
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()


class HW4RestorationDataset(Dataset):
    def __init__(self, root, split, split_name="train", patch_size=128, augment=True):
        self.root = Path(root)
        self.split_name = split_name
        self.patch_size = patch_size
        self.augment = augment and split_name == "train"
        self.degraded_dir = self.root / "train" / "degraded"
        self.clean_dir = self.root / "train" / "clean"
        self.items = []
        for task in ["rain", "snow"]:
            for name in split[split_name][task]:
                _, idx = parse_task_index(name)
                clean_name = f"{task}_clean-{idx}.png"
                self.items.append({
                    "task": task,
                    "task_id": 0 if task == "rain" else 1,
                    "degraded": name,
                    "clean": clean_name,
                })
        if split_name == "train":
            rng = random.Random(1234)
            rng.shuffle(self.items)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        degraded = pil_to_tensor(self.degraded_dir / item["degraded"])
        clean = pil_to_tensor(self.clean_dir / item["clean"])
        if self.split_name == "train":
            degraded, clean = self.random_crop_pair(degraded, clean, self.patch_size)
            if self.augment:
                if random.random() < 0.5:
                    degraded = torch.flip(degraded, dims=[2])
                    clean = torch.flip(clean, dims=[2])
                if random.random() < 0.5:
                    degraded = torch.flip(degraded, dims=[1])
                    clean = torch.flip(clean, dims=[1])

        meta = {"filename": item["degraded"], "task": item["task"], "task_id": item["task_id"]}
        return meta, degraded, clean

    @staticmethod
    def random_crop_pair(degraded, clean, patch_size):
        _, h, w = degraded.shape
        if h < patch_size or w < patch_size:
            pad_h = max(0, patch_size - h)
            pad_w = max(0, patch_size - w)
            mode = "reflect" if h > pad_h and w > pad_w else "replicate"
            degraded = F.pad(degraded.unsqueeze(0), (0, pad_w, 0, pad_h), mode=mode).squeeze(0)
            clean = F.pad(clean.unsqueeze(0), (0, pad_w, 0, pad_h), mode=mode).squeeze(0)
            _, h, w = degraded.shape
        top = random.randint(0, h - patch_size)
        left = random.randint(0, w - patch_size)
        return (
            degraded[:, top : top + patch_size, left : left + patch_size],
            clean[:, top : top + patch_size, left : left + patch_size],
        )


class HW4TestDataset(Dataset):
    def __init__(self, test_dir):
        self.test_dir = Path(test_dir)
        self.files = sorted_image_files(self.test_dir)
        if not self.files:
            raise FileNotFoundError(f"No PNG files found in {self.test_dir}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        tensor = pil_to_tensor(path)
        _, h, w = tensor.shape
        meta = {"filename": path.name, "height": h, "width": w}
        return meta, tensor


# Exact PromptIR model definitions are embedded in this cell.
# This keeps the notebook self-contained while preserving checkpoint key compatibility.

## PromptIR: Prompting for All-in-One Blind Image Restoration
## Vaishnav Potlapalli, Syed Waqas Zamir, Salman Khan, and Fahad Shahbaz Khan
## https://arxiv.org/abs/2306.13090


import torch
# print(torch.__version__)
import torch.nn as nn
import torch.nn.functional as F
import numbers

# Minimal local replacement for the small subset of einops.rearrange used here.
def rearrange(x, pattern, **kwargs):
    pattern = " ".join(pattern.split())
    if pattern == "b c h w -> b (h w) c":
        b, c, h, w = x.shape
        return x.permute(0, 2, 3, 1).reshape(b, h * w, c).contiguous()
    if pattern == "b (h w) c -> b c h w":
        h = kwargs["h"]
        w = kwargs["w"]
        b, hw, c = x.shape
        if hw != h * w:
            raise RuntimeError(f"Token count {hw} does not match h*w={h * w}")
        return x.reshape(b, h, w, c).permute(0, 3, 1, 2).contiguous()
    if pattern == "b (head c) h w -> b head c (h w)":
        head = kwargs["head"]
        b, hc, h, w = x.shape
        if hc % head != 0:
            raise RuntimeError(f"Channels {hc} are not divisible by heads {head}")
        c = hc // head
        return x.reshape(b, head, c, h * w).contiguous()
    if pattern == "b head c (h w) -> b (head c) h w":
        h = kwargs["h"]
        w = kwargs["w"]
        b, head, c, hw = x.shape
        if hw != h * w:
            raise RuntimeError(f"Token count {hw} does not match h*w={h * w}")
        return x.reshape(b, head * c, h, w).contiguous()
    raise NotImplementedError(f"Unsupported rearrange pattern: {pattern}")




##########################################################################
## Layer Norm

def to_3d(x):
    return rearrange(x, 'b c h w -> b (h w) c')

def to_4d(x,h,w):
    return rearrange(x, 'b (h w) c -> b c h w',h=h,w=w)

class BiasFree_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(BiasFree_LayerNorm, self).__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return x / torch.sqrt(sigma+1e-5) * self.weight
    




class WithBias_LayerNorm(nn.Module):
    def __init__(self, normalized_shape):
        super(WithBias_LayerNorm, self).__init__()
        if isinstance(normalized_shape, numbers.Integral):
            normalized_shape = (normalized_shape,)
        normalized_shape = torch.Size(normalized_shape)

        assert len(normalized_shape) == 1

        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.normalized_shape = normalized_shape

    def forward(self, x):
        mu = x.mean(-1, keepdim=True)
        sigma = x.var(-1, keepdim=True, unbiased=False)
        return (x - mu) / torch.sqrt(sigma+1e-5) * self.weight + self.bias


class LayerNorm(nn.Module):
    def __init__(self, dim, LayerNorm_type):
        super(LayerNorm, self).__init__()
        if LayerNorm_type =='BiasFree':
            self.body = BiasFree_LayerNorm(dim)
        else:
            self.body = WithBias_LayerNorm(dim)

    def forward(self, x):
        h, w = x.shape[-2:]
        return to_4d(self.body(to_3d(x)), h, w)



##########################################################################
## Gated-Dconv Feed-Forward Network (GDFN)
class FeedForward(nn.Module):
    def __init__(self, dim, ffn_expansion_factor, bias):
        super(FeedForward, self).__init__()

        hidden_features = int(dim*ffn_expansion_factor)

        self.project_in = nn.Conv2d(dim, hidden_features*2, kernel_size=1, bias=bias)

        self.dwconv = nn.Conv2d(hidden_features*2, hidden_features*2, kernel_size=3, stride=1, padding=1, groups=hidden_features*2, bias=bias)

        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x



##########################################################################
## Multi-DConv Head Transposed Self-Attention (MDTA)
class Attention(nn.Module):
    def __init__(self, dim, num_heads, bias):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        self.qkv = nn.Conv2d(dim, dim*3, kernel_size=1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim*3, dim*3, kernel_size=3, stride=1, padding=1, groups=dim*3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)
        


    def forward(self, x):
        b,c,h,w = x.shape

        qkv = self.qkv_dwconv(self.qkv(x))
        q,k,v = qkv.chunk(3, dim=1)   
        
        q = rearrange(q, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        k = rearrange(k, 'b (head c) h w -> b head c (h w)', head=self.num_heads)
        v = rearrange(v, 'b (head c) h w -> b head c (h w)', head=self.num_heads)

        q = torch.nn.functional.normalize(q, dim=-1)
        k = torch.nn.functional.normalize(k, dim=-1)

        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)

        out = (attn @ v)
        
        out = rearrange(out, 'b head c (h w) -> b (head c) h w', head=self.num_heads, h=h, w=w)

        out = self.project_out(out)
        return out



class resblock(nn.Module):
    def __init__(self, dim):

        super(resblock, self).__init__()
        # self.norm = LayerNorm(dim, LayerNorm_type='BiasFree')

        self.body = nn.Sequential(nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=False),
                                  nn.PReLU(),
                                  nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=1, bias=False))

    def forward(self, x):
        res = self.body((x))
        res += x
        return res


##########################################################################
## Resizing modules
class Downsample(nn.Module):
    def __init__(self, n_feat):
        super(Downsample, self).__init__()

        self.body = nn.Sequential(nn.Conv2d(n_feat, n_feat//2, kernel_size=3, stride=1, padding=1, bias=False),
                                  nn.PixelUnshuffle(2))

    def forward(self, x):
        return self.body(x)

class Upsample(nn.Module):
    def __init__(self, n_feat):
        super(Upsample, self).__init__()

        self.body = nn.Sequential(nn.Conv2d(n_feat, n_feat*2, kernel_size=3, stride=1, padding=1, bias=False),
                                  nn.PixelShuffle(2))

    def forward(self, x):
        return self.body(x)


##########################################################################
## Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, ffn_expansion_factor, bias, LayerNorm_type):
        super(TransformerBlock, self).__init__()

        self.norm1 = LayerNorm(dim, LayerNorm_type)
        self.attn = Attention(dim, num_heads, bias)
        self.norm2 = LayerNorm(dim, LayerNorm_type)
        self.ffn = FeedForward(dim, ffn_expansion_factor, bias)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))

        return x



##########################################################################
## Overlapped image patch embedding with 3x3 Conv
class OverlapPatchEmbed(nn.Module):
    def __init__(self, in_c=3, embed_dim=48, bias=False):
        super(OverlapPatchEmbed, self).__init__()

        self.proj = nn.Conv2d(in_c, embed_dim, kernel_size=3, stride=1, padding=1, bias=bias)

    def forward(self, x):
        x = self.proj(x)

        return x




##########################################################################
##---------- Prompt Gen Module -----------------------
class PromptGenBlock(nn.Module):
    """Original PromptIR prompt generation block.

    This block keeps the upstream PromptIR behavior for compatibility with the
    original denoise/derain/dehaze scripts.
    """

    def __init__(
        self,
        prompt_dim=128,
        prompt_len=5,
        prompt_size=96,
        lin_dim=192,
    ):
        super(PromptGenBlock, self).__init__()
        self.prompt_param = nn.Parameter(
            torch.rand(1, prompt_len, prompt_dim, prompt_size, prompt_size)
        )
        self.linear_layer = nn.Linear(lin_dim, prompt_len)
        self.conv3x3 = nn.Conv2d(
            prompt_dim,
            prompt_dim,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

    def forward(self, x, task_probs=None):
        del task_probs
        batch, channels, height, width = x.shape
        emb = x.mean(dim=(-2, -1))
        prompt_weights = F.softmax(self.linear_layer(emb), dim=1)
        prompt = (
            prompt_weights.unsqueeze(-1).unsqueeze(-1).unsqueeze(-1)
            * self.prompt_param.unsqueeze(0).repeat(batch, 1, 1, 1, 1, 1).squeeze(1)
        )
        prompt = torch.sum(prompt, dim=1)
        prompt = F.interpolate(prompt, (height, width), mode="bilinear")
        prompt = self.conv3x3(prompt)
        return prompt


class SoftTaskPromptGenBlock(nn.Module):
    """Prompt block with soft rain/snow prompt routing.

    The model remains a single PromptIR network. Instead of hard-selecting a
    rain or snow branch, a degradation gate predicts a soft probability over
    task-specific prompt banks. The selected bank is still mixed with the
    original content-adaptive prompt weights from PromptIR.
    """

    def __init__(
        self,
        prompt_dim=128,
        prompt_len=5,
        prompt_size=96,
        lin_dim=192,
        num_tasks=2,
    ):
        super(SoftTaskPromptGenBlock, self).__init__()
        self.num_tasks = num_tasks
        self.prompt_param = nn.Parameter(
            torch.rand(num_tasks, prompt_len, prompt_dim, prompt_size, prompt_size)
        )
        self.linear_layer = nn.Linear(lin_dim, prompt_len)
        self.conv3x3 = nn.Conv2d(
            prompt_dim,
            prompt_dim,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

    def forward(self, x, task_probs):
        if task_probs is None:
            task_probs = x.new_full((x.shape[0], self.num_tasks), 1.0 / self.num_tasks)

        batch, channels, height, width = x.shape
        del channels
        emb = x.mean(dim=(-2, -1))
        prompt_weights = F.softmax(self.linear_layer(emb), dim=1)

        task_mixed_prompts = torch.einsum(
            "bt,tnchw->bnchw", task_probs.to(x.dtype), self.prompt_param
        )
        prompt = task_mixed_prompts * prompt_weights[:, :, None, None, None]
        prompt = torch.sum(prompt, dim=1)
        prompt = F.interpolate(
            prompt,
            (height, width),
            mode="bilinear",
            align_corners=False,
        )
        prompt = self.conv3x3(prompt)
        return prompt


class DegradationGate(nn.Module):
    """Predicts soft rain/snow probabilities from latent PromptIR features."""

    def __init__(self, in_dim, num_tasks=2, hidden_dim=128):
        super(DegradationGate, self).__init__()
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_tasks),
        )

    def forward(self, x):
        return self.net(x)


class LocalWeatherRefinementHead(nn.Module):
    """Lightweight feature-space local weather refinement head.

    This module refines full-resolution decoder features before the final RGB
    output projection. It targets local rain streaks, snow particles, and
    residual weather artifacts with local, directional, and dilated branches.
    """

    def __init__(
        self,
        channels,
        expansion=2,
        residual_scale_init=0.1,
        zero_init_final=True,
    ):
        super(LocalWeatherRefinementHead, self).__init__()
        hidden_channels = int(channels * expansion)

        self.pre = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, kernel_size=1, bias=True),
            nn.GELU(),
        )
        self.dw_3x3 = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=3,
            padding=1,
            groups=hidden_channels,
            bias=True,
        )
        self.dw_1x7 = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(1, 7),
            padding=(0, 3),
            groups=hidden_channels,
            bias=True,
        )
        self.dw_7x1 = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=(7, 1),
            padding=(3, 0),
            groups=hidden_channels,
            bias=True,
        )
        self.dw_dilated = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=3,
            padding=2,
            dilation=2,
            groups=hidden_channels,
            bias=True,
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(hidden_channels * 4, hidden_channels, kernel_size=1, bias=True),
            nn.GELU(),
        )
        self.out = nn.Conv2d(hidden_channels, channels, kernel_size=1, bias=True)
        self.residual_scale = nn.Parameter(
            torch.ones(1, dtype=torch.float32) * float(residual_scale_init)
        )

        if zero_init_final:
            nn.init.zeros_(self.out.weight)
            if self.out.bias is not None:
                nn.init.zeros_(self.out.bias)

    def forward(self, x):
        feat = self.pre(x)
        b1 = self.dw_3x3(feat)
        b2 = self.dw_1x7(feat)
        b3 = self.dw_7x1(feat)
        b4 = self.dw_dilated(feat)
        fused = self.fuse(torch.cat([b1, b2, b3, b4], dim=1))
        delta = self.out(fused)
        return x + self.residual_scale.to(delta.dtype) * delta


##########################################################################
##---------- PromptIR -----------------------


class PromptIR(nn.Module):
    def __init__(
        self,
        inp_channels=3,
        out_channels=3,
        dim=48,
        num_blocks=[4, 6, 6, 8],
        num_refinement_blocks=4,
        prompt_len=5,
        heads=[1, 2, 4, 8],
        ffn_expansion_factor=2.66,
        bias=False,
        LayerNorm_type='WithBias',
        decoder=False,
        use_task_prompt_routing=False,
        num_tasks=2,
        use_local_weather_refine=False,
        lwr_expansion=2,
        lwr_res_scale_init=0.1,
        lwr_zero_init_final=True,
    ):
        super(PromptIR, self).__init__()

        self.patch_embed = OverlapPatchEmbed(inp_channels, dim)
        self.decoder = decoder
        self.use_task_prompt_routing = use_task_prompt_routing
        self.num_tasks = num_tasks
        self.use_local_weather_refine = use_local_weather_refine
        level1_channels = dim
        level2_channels = int(dim * 2 ** 1)
        level3_channels = int(dim * 2 ** 2)
        latent_channels = int(dim * 2 ** 3)
        refinement_channels = level2_channels

        if self.decoder:
            prompt_block = SoftTaskPromptGenBlock if use_task_prompt_routing else PromptGenBlock
            common_kwargs = {"num_tasks": num_tasks} if use_task_prompt_routing else {}
            self.prompt1 = prompt_block(
                prompt_dim=64,
                prompt_len=prompt_len,
                prompt_size=64,
                lin_dim=level2_channels,
                **common_kwargs,
            )
            self.prompt2 = prompt_block(
                prompt_dim=128,
                prompt_len=prompt_len,
                prompt_size=32,
                lin_dim=level3_channels,
                **common_kwargs,
            )
            self.prompt3 = prompt_block(
                prompt_dim=320,
                prompt_len=prompt_len,
                prompt_size=16,
                lin_dim=latent_channels,
                **common_kwargs,
            )
            if use_task_prompt_routing:
                self.degradation_gate = DegradationGate(
                    in_dim=latent_channels,
                    num_tasks=num_tasks,
                )
            else:
                self.degradation_gate = None
        else:
            self.degradation_gate = None

        self.chnl_reduce1 = nn.Conv2d(64, 64, kernel_size=1, bias=bias)
        self.chnl_reduce2 = nn.Conv2d(128, 128, kernel_size=1, bias=bias)
        self.chnl_reduce3 = nn.Conv2d(320, 256, kernel_size=1, bias=bias)

        self.reduce_noise_channel_1 = nn.Conv2d(
            level1_channels + 64,
            level1_channels,
            kernel_size=1,
            bias=bias,
        )
        self.encoder_level1 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=level1_channels,
                    num_heads=heads[0],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[0])
            ]
        )

        self.down1_2 = Downsample(level1_channels)

        self.reduce_noise_channel_2 = nn.Conv2d(
            level2_channels + 128,
            level2_channels,
            kernel_size=1,
            bias=bias,
        )
        self.encoder_level2 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=level2_channels,
                    num_heads=heads[1],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[1])
            ]
        )

        self.down2_3 = Downsample(level2_channels)

        self.reduce_noise_channel_3 = nn.Conv2d(
            level3_channels + 256,
            level3_channels,
            kernel_size=1,
            bias=bias,
        )
        self.encoder_level3 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=level3_channels,
                    num_heads=heads[2],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[2])
            ]
        )

        self.down3_4 = Downsample(level3_channels)
        self.latent = nn.Sequential(
            *[
                TransformerBlock(
                    dim=latent_channels,
                    num_heads=heads[3],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[3])
            ]
        )

        self.up4_3 = Upsample(level3_channels)
        self.reduce_chan_level3 = nn.Conv2d(
            level2_channels + level3_channels,
            level3_channels,
            kernel_size=1,
            bias=bias,
        )
        self.noise_level3 = TransformerBlock(
            dim=latent_channels + 320,
            num_heads=heads[2],
            ffn_expansion_factor=ffn_expansion_factor,
            bias=bias,
            LayerNorm_type=LayerNorm_type,
        )
        self.reduce_noise_level3 = nn.Conv2d(
            latent_channels + 320,
            level3_channels,
            kernel_size=1,
            bias=bias,
        )

        self.decoder_level3 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=level3_channels,
                    num_heads=heads[2],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[2])
            ]
        )

        self.up3_2 = Upsample(level3_channels)
        self.reduce_chan_level2 = nn.Conv2d(
            level3_channels,
            level2_channels,
            kernel_size=1,
            bias=bias,
        )
        self.noise_level2 = TransformerBlock(
            dim=level3_channels + 128,
            num_heads=heads[2],
            ffn_expansion_factor=ffn_expansion_factor,
            bias=bias,
            LayerNorm_type=LayerNorm_type,
        )
        self.reduce_noise_level2 = nn.Conv2d(
            level3_channels + 128,
            level3_channels,
            kernel_size=1,
            bias=bias,
        )

        self.decoder_level2 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=level2_channels,
                    num_heads=heads[1],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[1])
            ]
        )

        self.up2_1 = Upsample(level2_channels)

        self.noise_level1 = TransformerBlock(
            dim=level2_channels + 64,
            num_heads=heads[2],
            ffn_expansion_factor=ffn_expansion_factor,
            bias=bias,
            LayerNorm_type=LayerNorm_type,
        )
        self.reduce_noise_level1 = nn.Conv2d(
            level2_channels + 64,
            level2_channels,
            kernel_size=1,
            bias=bias,
        )

        self.decoder_level1 = nn.Sequential(
            *[
                TransformerBlock(
                    dim=refinement_channels,
                    num_heads=heads[0],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_blocks[0])
            ]
        )

        self.refinement = nn.Sequential(
            *[
                TransformerBlock(
                    dim=refinement_channels,
                    num_heads=heads[0],
                    ffn_expansion_factor=ffn_expansion_factor,
                    bias=bias,
                    LayerNorm_type=LayerNorm_type,
                )
                for _ in range(num_refinement_blocks)
            ]
        )

        if self.use_local_weather_refine:
            self.local_weather_refine = LocalWeatherRefinementHead(
                channels=refinement_channels,
                expansion=lwr_expansion,
                residual_scale_init=lwr_res_scale_init,
                zero_init_final=lwr_zero_init_final,
            )

        self.output = nn.Conv2d(
            refinement_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=bias,
        )

    def forward(self, inp_img, noise_emb=None, return_gate=False):
        del noise_emb
        gate_logits = None
        task_probs = None

        inp_enc_level1 = self.patch_embed(inp_img)
        out_enc_level1 = self.encoder_level1(inp_enc_level1)

        inp_enc_level2 = self.down1_2(out_enc_level1)
        out_enc_level2 = self.encoder_level2(inp_enc_level2)

        inp_enc_level3 = self.down2_3(out_enc_level2)
        out_enc_level3 = self.encoder_level3(inp_enc_level3)

        inp_enc_level4 = self.down3_4(out_enc_level3)
        latent = self.latent(inp_enc_level4)

        if self.decoder:
            if self.use_task_prompt_routing and self.degradation_gate is not None:
                gate_logits = self.degradation_gate(latent)
                task_probs = F.softmax(gate_logits, dim=1)

            dec3_param = self.prompt3(latent, task_probs=task_probs)
            latent = torch.cat([latent, dec3_param], 1)
            latent = self.noise_level3(latent)
            latent = self.reduce_noise_level3(latent)

        inp_dec_level3 = self.up4_3(latent)
        inp_dec_level3 = torch.cat([inp_dec_level3, out_enc_level3], 1)
        inp_dec_level3 = self.reduce_chan_level3(inp_dec_level3)

        out_dec_level3 = self.decoder_level3(inp_dec_level3)
        if self.decoder:
            dec2_param = self.prompt2(out_dec_level3, task_probs=task_probs)
            out_dec_level3 = torch.cat([out_dec_level3, dec2_param], 1)
            out_dec_level3 = self.noise_level2(out_dec_level3)
            out_dec_level3 = self.reduce_noise_level2(out_dec_level3)

        inp_dec_level2 = self.up3_2(out_dec_level3)
        inp_dec_level2 = torch.cat([inp_dec_level2, out_enc_level2], 1)
        inp_dec_level2 = self.reduce_chan_level2(inp_dec_level2)

        out_dec_level2 = self.decoder_level2(inp_dec_level2)
        if self.decoder:
            dec1_param = self.prompt1(out_dec_level2, task_probs=task_probs)
            out_dec_level2 = torch.cat([out_dec_level2, dec1_param], 1)
            out_dec_level2 = self.noise_level1(out_dec_level2)
            out_dec_level2 = self.reduce_noise_level1(out_dec_level2)

        inp_dec_level1 = self.up2_1(out_dec_level2)
        inp_dec_level1 = torch.cat([inp_dec_level1, out_enc_level1], 1)

        out_dec_level1 = self.decoder_level1(inp_dec_level1)
        out_dec_level1 = self.refinement(out_dec_level1)
        if self.use_local_weather_refine:
            out_dec_level1 = self.local_weather_refine(out_dec_level1)
        out_dec_level1 = self.output(out_dec_level1) + inp_img

        if return_gate:
            return out_dec_level1, gate_logits
        return out_dec_level1

def stage_get(cfg, stage, key, default=None):
    if stage is not None and key in stage:
        return stage[key]
    return getattr(cfg, key, default)


def stage_uses_lwr(cfg, stage=None):
    return bool(stage_get(cfg, stage, "USE_LOCAL_WEATHER_REFINE", False))


def stage_allows_lwr_partial_load(cfg, stage=None):
    return bool(stage_get(cfg, stage, "ALLOW_PARTIAL_LOAD_FOR_LWR", False))


def build_model(cfg, stage=None):
    if cfg.USE_SOFT_PROMPT_ROUTING:
        raise NotImplementedError("Soft prompt routing is intentionally disabled for this Kaggle notebook default.")
    return PromptIR(
        decoder=True,
        dim=cfg.MODEL_DIM,
        num_blocks=cfg.NUM_BLOCKS,
        num_refinement_blocks=cfg.NUM_REFINEMENT_BLOCKS,
        prompt_len=cfg.PROMPT_LEN,
        heads=cfg.NUM_HEADS,
        ffn_expansion_factor=cfg.FFN_EXPANSION_FACTOR,
        bias=cfg.BIAS,
        LayerNorm_type=cfg.LAYER_NORM_TYPE,
        use_task_prompt_routing=False,
        num_tasks=2,
        use_local_weather_refine=stage_uses_lwr(cfg, stage),
        lwr_expansion=stage_get(cfg, stage, "LWR_EXPANSION", cfg.LWR_EXPANSION),
        lwr_res_scale_init=stage_get(cfg, stage, "LWR_RES_SCALE_INIT", cfg.LWR_RES_SCALE_INIT),
        lwr_zero_init_final=stage_get(cfg, stage, "LWR_ZERO_INIT_FINAL", cfg.LWR_ZERO_INIT_FINAL),
    )

def freeze_known_unused_params(model):
    unused_prefixes = (
        "chnl_reduce1.",
        "chnl_reduce2.",
        "chnl_reduce3.",
        "reduce_noise_channel_1.",
        "reduce_noise_channel_2.",
        "reduce_noise_channel_3.",
    )
    frozen = []
    for name, param in model.named_parameters():
        if name.startswith(unused_prefixes):
            param.requires_grad = False
            frozen.append(name)
    if frozen:
        log_rank0("Frozen known-unused params:", frozen)
    return frozen

    
def model_summary(model, cfg, device="cuda", stage=None):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Model config:", {
        "dim": cfg.MODEL_DIM,
        "num_blocks": cfg.NUM_BLOCKS,
        "num_refinement_blocks": cfg.NUM_REFINEMENT_BLOCKS,
        "prompt_len": cfg.PROMPT_LEN,
        "use_local_weather_refine": stage_uses_lwr(cfg, stage),
        "lwr_expansion": stage_get(cfg, stage, "LWR_EXPANSION", cfg.LWR_EXPANSION),
        "lwr_res_scale_init": stage_get(cfg, stage, "LWR_RES_SCALE_INIT", cfg.LWR_RES_SCALE_INIT),
    })
    print(f"Total params: {total / 1e6:.3f}M")
    print(f"Trainable params: {trainable / 1e6:.3f}M")
    model.eval()
    with torch.no_grad():
        x = torch.randn(1, 3, 64, 64, device=device)
        y = model.to(device)(x)
    print("Dummy input/output:", tuple(x.shape), tuple(y.shape))


def strip_prefix_if_present(state, prefix):
    if all(k.startswith(prefix) for k in state.keys()):
        return {k[len(prefix):]: v for k, v in state.items()}
    return state


def is_expected_lwr_missing_key(key):
    return key.startswith("local_weather_refine.")


def load_checkpoint(model, ckpt_path, map_location="cpu", allow_partial_load_for_lwr=False):
    ckpt = torch.load(ckpt_path, map_location=map_location)
    if isinstance(ckpt, dict) and "ema_state_dict" in ckpt:
        state = ckpt["ema_state_dict"]
        log_rank0("Using ema_state_dict from checkpoint")
    elif isinstance(ckpt, dict) and "model" in ckpt:
        state = ckpt["model"]
        log_rank0("Using model state from checkpoint")
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state = ckpt["state_dict"]
        log_rank0("Using Lightning state_dict from checkpoint")
    elif isinstance(ckpt, dict):
        state = ckpt
        log_rank0("Using raw state_dict checkpoint")
    else:
        raise TypeError(f"Unsupported checkpoint type: {type(ckpt)}")
    state = strip_prefix_if_present(state, "module.")
    state = strip_prefix_if_present(state, "net.")
    missing, unexpected = model.load_state_dict(state, strict=False)
    missing = list(missing)
    unexpected = list(unexpected)
    if missing or unexpected:
        log_rank0("Checkpoint load missing keys:")
        if missing:
            for key in missing:
                log_rank0("  ", key)
        else:
            log_rank0("   <none>")
        log_rank0("Checkpoint load unexpected keys:")
        if unexpected:
            for key in unexpected:
                log_rank0("  ", key)
        else:
            log_rank0("   <none>")

    if allow_partial_load_for_lwr:
        serious_missing = [key for key in missing if not is_expected_lwr_missing_key(key)]
        if serious_missing:
            log_rank0("WARNING: Non-LWR keys are missing. Check checkpoint compatibility.")
            raise RuntimeError(f"Non-LWR missing checkpoint keys: {serious_missing[:10]}")
        if unexpected:
            raise RuntimeError(f"Unexpected checkpoint keys: {unexpected[:10]}")
    elif missing or unexpected:
        raise RuntimeError(f"Checkpoint mismatch. Missing={missing[:5]}, unexpected={unexpected[:5]}")
    return model


def save_training_object(obj, path, cfg):
    if wants_tpu(cfg):
        xm_mod, _, _ = require_xla()
        xm_mod.save(obj, str(path))
    else:
        torch.save(obj, path)


def save_checkpoint(path, model, stage, epoch, val_psnr, cfg):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
    payload = {
        "model": state,
        "stage": stage,
        "epoch": epoch,
        "val_psnr": float(val_psnr),
        "config": {
            "MODEL_DIM": cfg.MODEL_DIM,
            "NUM_BLOCKS": cfg.NUM_BLOCKS,
            "NUM_REFINEMENT_BLOCKS": cfg.NUM_REFINEMENT_BLOCKS,
            "PROMPT_LEN": cfg.PROMPT_LEN,
        },
    }
    save_training_object(payload, path, cfg)


def save_full_checkpoint(path, model, optimizer, scheduler, scaler, stage, epoch, val_psnr, cfg, global_step=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()

    payload = {
        "model": state,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "stage": stage,
        "epoch": epoch,
        "global_step": global_step,
        "val_psnr": float(val_psnr),
        "config": {
            k: getattr(cfg, k)
            for k in dir(cfg)
            if k.isupper()
        },
        "torch_rng_state": torch.get_rng_state(),
    }

    if torch.cuda.is_available():
        payload["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()

    save_training_object(payload, path, cfg)

def pad_to_multiple(x, multiple=8):
    _, _, h, w = x.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    if pad_h == 0 and pad_w == 0:
        return x, (h, w)
    mode = "reflect" if h > pad_h and w > pad_w else "replicate"
    x = F.pad(x, (0, pad_w, 0, pad_h), mode=mode)
    return x, (h, w)


def unpad(x, size):
    h, w = size
    return x[..., :h, :w]


def tile_starts(length, tile_size, stride):
    if length <= tile_size:
        return [0]
    starts = list(range(0, length - tile_size + 1, stride))
    if starts[-1] != length - tile_size:
        starts.append(length - tile_size)
    return starts


def model_forward(model, x):
    y = model(x)
    if isinstance(y, (tuple, list)):
        y = y[0]
    return y


def tiled_forward(model, x, tile_size=256, tile_overlap=32):
    b, c, h, w = x.shape
    stride = max(1, tile_size - tile_overlap)
    ys = tile_starts(h, tile_size, stride)
    xs = tile_starts(w, tile_size, stride)
    output = torch.zeros_like(x)
    weight = torch.zeros_like(x)
    for y0 in ys:
        for x0 in xs:
            tile = x[..., y0 : y0 + tile_size, x0 : x0 + tile_size]
            pred = model_forward(model, tile)
            output[..., y0 : y0 + tile.shape[-2], x0 : x0 + tile.shape[-1]] += pred
            weight[..., y0 : y0 + tile.shape[-2], x0 : x0 + tile.shape[-1]] += 1
    return output / weight.clamp_min(1)


def forward_with_padding(model, x, multiple=8, use_tile=False, tile_size=256, tile_overlap=32):
    x_pad, original_size = pad_to_multiple(x, multiple)
    if use_tile:
        y = tiled_forward(model, x_pad, tile_size, tile_overlap)
    else:
        y = model_forward(model, x_pad)
    return unpad(y, original_size)


def tta_transforms(x):
    return [
        (x, lambda y: y),
        (torch.flip(x, [-1]), lambda y: torch.flip(y, [-1])),
        (torch.flip(x, [-2]), lambda y: torch.flip(y, [-2])),
        (torch.flip(x, [-2, -1]), lambda y: torch.flip(y, [-2, -1])),
        (x.transpose(-2, -1), lambda y: y.transpose(-2, -1)),
        (torch.flip(x.transpose(-2, -1), [-1]), lambda y: torch.flip(y, [-1]).transpose(-2, -1)),
        (torch.flip(x.transpose(-2, -1), [-2]), lambda y: torch.flip(y, [-2]).transpose(-2, -1)),
        (torch.flip(x.transpose(-2, -1), [-2, -1]), lambda y: torch.flip(y, [-2, -1]).transpose(-2, -1)),
    ]


def predict_with_tta(model, x, use_tile=True, tile_size=256, tile_overlap=32):
    preds = []
    for aug, inv in tta_transforms(x):
        pred = forward_with_padding(model, aug, use_tile=use_tile, tile_size=tile_size, tile_overlap=tile_overlap)
        preds.append(inv(pred))
    return torch.stack(preds, dim=0).mean(dim=0)


def psnr_tensor(pred, target):
    pred = pred.clamp(0, 1)
    target = target.clamp(0, 1)
    mse = (pred - target).pow(2).flatten(1).mean(dim=1)
    psnr = -10.0 * torch.log10(mse.clamp_min(1e-12))
    return psnr


@torch.no_grad()
def validate_model(model, dataset, device, use_tile=False, tile_size=256, tile_overlap=32, use_tta=False, amp=True, show_progress=True):
    model.eval()
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    all_scores = []
    by_task = {"rain": [], "snow": []}
    pbar = tqdm(
        loader,
        total=len(loader),
        disable=(not is_rank0()) or (not show_progress),
        dynamic_ncols=True,
        leave=False,
        desc="validation",
    ) if tqdm is not None else loader
    for meta, degraded, clean in pbar:
        degraded = degraded.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp and device.type == "cuda"):
            if use_tta:
                pred = predict_with_tta(model, degraded, use_tile=use_tile, tile_size=tile_size, tile_overlap=tile_overlap)
            else:
                pred = forward_with_padding(model, degraded, use_tile=use_tile, tile_size=tile_size, tile_overlap=tile_overlap)
        score = psnr_tensor(pred.float(), clean.float()).item()
        task = meta["task"][0]
        all_scores.append(score)
        by_task[task].append(score)
    return {
        "overall": float(np.mean(all_scores)),
        "rain": float(np.mean(by_task["rain"])),
        "snow": float(np.mean(by_task["snow"])),
    }


def make_scheduler(
    optimizer,
    warmup_steps,
    total_steps,
    scheduler_type="cosine",
    eta_min=0.0,
):
    """Build a per-step warmup + cosine annealing LR scheduler."""
    scheduler_type = str(scheduler_type).lower()
    total_steps = max(1, int(total_steps))
    warmup_steps = max(0, int(warmup_steps))
    eta_min = max(0.0, float(eta_min))

    if scheduler_type not in {"cosine", "cosine_annealing"}:
        raise ValueError(f"Unsupported LR_SCHEDULER={scheduler_type!r}; use 'cosine'.")

    def make_group_lambda(base_lr):
        min_factor = min(1.0, eta_min / float(base_lr)) if base_lr > 0 else 0.0

        def lr_lambda(step):
            if warmup_steps > 0 and step < warmup_steps:
                return float(step + 1) / float(max(1, warmup_steps))
            progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
            progress = min(1.0, max(0.0, progress))
            cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
            return min_factor + (1.0 - min_factor) * cosine_factor

        return lr_lambda

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=[make_group_lambda(group["lr"]) for group in optimizer.param_groups],
    )


def stage_dir(cfg, stage_name):
    return Path(cfg.OUTPUT_DIR) / "runs" / cfg.RUN_NAME / stage_name


def normalize_run_only_stage(value):
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.lower() in {"", "none", "null", "all"}:
            return None
        return stripped
    return value


def get_stage_by_name(cfg, stage_name):
    stage_name = normalize_run_only_stage(stage_name)
    if stage_name is None:
        return None
    available = [stage.get("NAME") for stage in getattr(cfg, "STAGES", [])]
    for stage in cfg.STAGES:
        if stage.get("NAME") == stage_name:
            return stage
    raise ValueError(
        f"Unknown stage: {stage_name}. Available stages: {available}. "
        "Set RUN_ONLY_STAGE to one of these names, or set RUN_ONLY_STAGE=None."
    )


def stage_for_checkpoint(cfg, ckpt_path):
    explicit_stage = normalize_run_only_stage(getattr(cfg, "INFERENCE_STAGE", None))
    if explicit_stage is not None:
        return get_stage_by_name(cfg, explicit_stage)
    ckpt_path = Path(ckpt_path).resolve()
    for stage in cfg.STAGES:
        try:
            if stage_dir(cfg, stage["NAME"]).resolve() in ckpt_path.parents:
                return stage
        except Exception:
            continue
    run_only = normalize_run_only_stage(getattr(cfg, "RUN_ONLY_STAGE", None))
    if run_only is not None:
        return get_stage_by_name(cfg, run_only)
    return None


def _checkpoint_score(path):
    name = Path(path).stem
    if "psnr" in name:
        try:
            return float(name.split("psnr")[-1])
        except ValueError:
            return -1.0
    return -1.0


def find_best_checkpoint(path):
    path = Path(path)
    candidates = sorted(path.glob("epoch*-psnr*.ckpt"))
    if not candidates:
        candidates = sorted(path.glob("best*.ckpt"))
    if not candidates:
        return None
    return max(candidates, key=_checkpoint_score)


def _valid_checkpoint_path(value):
    if value is None:
        return None
    if isinstance(value, str) and value.strip().lower() in {"", "none", "null"}:
        return None
    path = Path(value)
    return path if path.exists() else None


def find_best_checkpoint_from_globs(globs):
    candidates = []
    for pattern in globs or []:
        for item in glob.glob(str(pattern), recursive=True):
            path = Path(item)
            if path.is_file() and path.suffix == ".ckpt":
                candidates.append(path)
    if not candidates:
        return None
    return max(sorted(set(candidates)), key=_checkpoint_score)


def resolve_named_checkpoint(cfg, name):
    name = str(name).upper()
    if name == "BEST_192":
        explicit = _valid_checkpoint_path(getattr(cfg, "BEST_192_CKPT_PATH", None))
        if explicit is not None:
            log_rank0("Resolved BEST_192_CKPT_PATH:", explicit)
            return explicit
        ckpt = find_best_checkpoint_from_globs(getattr(cfg, "BEST_192_CKPT_GLOBS", []))
        if ckpt is not None:
            log_rank0("Resolved BEST_192 from globs:", ckpt)
            return ckpt
        raise FileNotFoundError(
            "Could not resolve AUTO_BEST_192. Set BEST_192_CKPT_PATH to your "
            "uploaded 192 checkpoint, or update BEST_192_CKPT_GLOBS."
        )
    if name == "BEST_224":
        explicit = _valid_checkpoint_path(getattr(cfg, "BEST_224_CKPT_PATH", None))
        if explicit is not None:
            log_rank0("Resolved BEST_224_CKPT_PATH:", explicit)
            return explicit
        ckpt = find_best_checkpoint_from_globs(getattr(cfg, "BEST_224_CKPT_GLOBS", []))
        if ckpt is not None:
            log_rank0("Resolved BEST_224 from globs:", ckpt)
            return ckpt
        raise FileNotFoundError(
            "Could not resolve AUTO_BEST_224. Set BEST_224_CKPT_PATH to your "
            "uploaded 224 checkpoint, or update BEST_224_CKPT_GLOBS."
        )
    raise ValueError(f"Unknown named checkpoint alias: {name}")


def resolve_stage_checkpoint(cfg, stage_name):
    stage_name = normalize_run_only_stage(stage_name)
    available = [stage.get("NAME") for stage in getattr(cfg, "STAGES", [])]
    if stage_name in available:
        stage = get_stage_by_name(cfg, stage_name)
        search_name = stage["NAME"]
    else:
        search_name = stage_name
        log_rank0(
            f"AUTO_BEST_STAGE:{stage_name} is not in configured STAGES={available}; "
            f"searching output stage directory directly."
        )
    ckpt = find_best_checkpoint(stage_dir(cfg, search_name))
    if ckpt is None:
        best = stage_dir(cfg, search_name) / "best.ckpt"
        ckpt = best if best.exists() else None
    if ckpt is None:
        raise FileNotFoundError(
            f"Could not find best checkpoint for stage {stage_name} under {stage_dir(cfg, search_name)}. "
            "For standalone LWR, set the LWR stage INIT_CKPT='AUTO_BEST_224' and BEST_224_CKPT_PATH to the uploaded 224 checkpoint."
        )
    log_rank0(f"Resolved AUTO_BEST_STAGE:{stage_name} -> {ckpt}")
    return ckpt


def resolve_init_checkpoint(cfg, stage, previous_best):
    if cfg.RESUME_CHECKPOINT:
        ckpt = Path(cfg.RESUME_CHECKPOINT)
        if not ckpt.exists():
            raise FileNotFoundError(f"RESUME_CHECKPOINT does not exist: {ckpt}")
        return str(ckpt)
    init = stage.get("INIT_CKPT")
    if init is None:
        return None
    if init == "AUTO_BEST_PREVIOUS":
        if previous_best:
            ckpt = Path(previous_best)
        else:
            stage_names = [s["NAME"] for s in cfg.STAGES]
            idx = stage_names.index(stage["NAME"])
            if idx <= 0:
                raise ValueError(
                    f"Stage {stage['NAME']} requested AUTO_BEST_PREVIOUS, "
                    "but it is the first configured stage."
                )
            ckpt = resolve_stage_checkpoint(cfg, stage_names[idx - 1])
        if ckpt is None or not Path(ckpt).exists():
            raise FileNotFoundError(f"AUTO_BEST_PREVIOUS for {stage['NAME']} could not find a checkpoint")
        log_rank0(f"Resolved AUTO_BEST_PREVIOUS for {stage['NAME']}: {ckpt}")
        return str(ckpt)
    if isinstance(init, str) and init.startswith("AUTO_BEST_STAGE:"):
        stage_name = init.split(":", 1)[1]
        return str(resolve_stage_checkpoint(cfg, stage_name))
    if init == "AUTO_BEST_192":
        return str(resolve_named_checkpoint(cfg, "BEST_192"))
    if init == "AUTO_BEST_224":
        return str(resolve_named_checkpoint(cfg, "BEST_224"))
    ckpt = Path(init)
    if not ckpt.exists():
        raise FileNotFoundError(f"INIT_CKPT does not exist for stage {stage['NAME']}: {ckpt}")
    return str(ckpt)


def sync_stop_flag(stop_flag, device, distributed=False, is_xla=False):
    value = torch.tensor([1 if stop_flag else 0], device=device, dtype=torch.int32)
    if is_xla and distributed:
        xm_mod, _, _ = require_xla()
        value = xm_mod.all_reduce(xm_mod.REDUCE_MAX, value)
        xm_mod.mark_step()
    elif distributed:
        dist.all_reduce(value, op=dist.ReduceOp.MAX)
    return bool(int(value.item()))

def _run_training_xla_worker(index):
    del index
    import hw4_config as cfg
    configure_tpu_environment(cfg)
    xm_mod, _, _ = require_xla()
    device = xm_mod.xla_device()
    local_rank = xm_mod.get_ordinal()
    world_size = xm_mod.xrt_world_size()
    _run_training_impl(
        cfg,
        device=device,
        local_rank=local_rank,
        world_size=world_size,
        distributed=world_size > 1,
        is_xla=True,
    )


def run_training(cfg):
    if wants_tpu(cfg):
        configure_tpu_environment(cfg)
        _, xmp_mod, _ = require_xla()
        nprocs = resolve_tpu_nprocs(cfg)
        print(
            "TPU runtime env:",
            {
                "PJRT_DEVICE": os.environ.get("PJRT_DEVICE"),
                "TPU_PROCESS_BOUNDS": os.environ.get("TPU_PROCESS_BOUNDS"),
                "TPU_VISIBLE_CHIPS": os.environ.get("TPU_VISIBLE_CHIPS"),
                "XLA_USE_BF16": os.environ.get("XLA_USE_BF16"),
                "nprocs": nprocs if nprocs is not None else "AUTO",
            },
            flush=True,
        )
        if nprocs == 1:
            print("Launching PyTorch/XLA TPU training in single-process mode", flush=True)
            _run_training_xla_worker(0)
        elif nprocs is None:
            print("Launching PyTorch/XLA TPU training with xmp.spawn(nprocs=AUTO)", flush=True)
            xmp_mod.spawn(_run_training_xla_worker, args=(), start_method="fork")
        else:
            print(f"Launching PyTorch/XLA TPU training with xmp.spawn(nprocs={nprocs})", flush=True)
            xmp_mod.spawn(_run_training_xla_worker, args=(), nprocs=nprocs, start_method="fork")
        return
    return _run_training_impl(cfg)


def _run_training_impl(cfg, device=None, local_rank=None, world_size=None, distributed=None, is_xla=False):
    torch.set_float32_matmul_precision(cfg.MATMUL_PRECISION)
    if device is None:
        local_rank = int(os.environ.get("LOCAL_RANK", "0"))
        world_size = int(os.environ.get("WORLD_SIZE", "1"))
        distributed = world_size > 1
        if torch.cuda.is_available():
            torch.cuda.set_device(local_rank)
            device = torch.device("cuda", local_rank)
        else:
            device = torch.device("cpu")
        if distributed:
            dist.init_process_group(backend="nccl")
        is_xla = False
    else:
        local_rank = int(local_rank or 0)
        world_size = int(world_size or 1)
        distributed = bool(distributed)
    seed_everything(cfg.SEED + local_rank)

    root, split = build_or_load_split(cfg)
    if is_rank0():
        Path(cfg.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        shutil.copy2(cfg.SPLIT_JSON_PATH, Path(cfg.OUTPUT_DIR) / "split.json")
        Path(cfg.OUTPUT_DIR, "config.json").write_text(json.dumps({
            k: getattr(cfg, k) for k in dir(cfg) if k.isupper()
        }, indent=2, default=str), encoding="utf-8")
        log_rank0("DATA_ROOT:", root)
        log_rank0("WORLD_SIZE:", world_size)

    stages = cfg.STAGES
    run_only_stage = normalize_run_only_stage(cfg.RUN_ONLY_STAGE)
    if run_only_stage is not None:
        stages = [s for s in stages if s["NAME"] == run_only_stage]
        if not stages:
            raise ValueError(f"RUN_ONLY_STAGE not found: {cfg.RUN_ONLY_STAGE}")

    previous_best = None
    for stage in stages:
        stage_name = stage["NAME"]
        out_dir = stage_dir(cfg, stage_name)
        if is_rank0():
            out_dir.mkdir(parents=True, exist_ok=True)
            log_rank0(f"\n=== Stage {stage_name} ===")

        train_set = HW4RestorationDataset(root, split, "train", patch_size=stage["PATCH_SIZE"], augment=True)
        val_set = HW4RestorationDataset(root, split, "val", patch_size=stage["PATCH_SIZE"], augment=False)
        if distributed:
            sampler = DistributedSampler(train_set, num_replicas=world_size, rank=local_rank, shuffle=True)
        else:
            sampler = None
        loader = DataLoader(
            train_set,
            batch_size=stage["BATCH_SIZE_PER_GPU"],
            shuffle=sampler is None,
            sampler=sampler,
            num_workers=cfg.NUM_WORKERS,
            pin_memory=cfg.PIN_MEMORY and device.type == "cuda",
            persistent_workers=cfg.PERSISTENT_WORKERS and cfg.NUM_WORKERS > 0,
            drop_last=True,
        )
        train_loader = loader
        if is_xla:
            _, _, xla_pl_mod = require_xla()
            train_loader = xla_pl_mod.MpDeviceLoader(loader, device)

        stage_use_lwr = stage_uses_lwr(cfg, stage)
        stage_allow_lwr_partial = stage_allows_lwr_partial_load(cfg, stage)
        model = build_model(cfg, stage).to(device)
        init_ckpt = resolve_init_checkpoint(cfg, stage, previous_best)
        if init_ckpt:
            log_rank0("Loading init checkpoint:", init_ckpt)
            load_checkpoint(
                model,
                init_ckpt,
                map_location=("cpu" if is_xla else device),
                allow_partial_load_for_lwr=stage_use_lwr and stage_allow_lwr_partial,
            )

        freeze_known_unused_params(model)
        
        if is_rank0():
            model_summary(model, cfg, device=device, stage=stage)
            log_rank0("Batch per GPU:", stage["BATCH_SIZE_PER_GPU"])
            log_rank0("Effective batch:", stage["BATCH_SIZE_PER_GPU"] * world_size)
            log_rank0("LWR enabled:", stage_use_lwr)
            if stage_use_lwr:
                log_rank0("LWR expansion:", stage_get(cfg, stage, "LWR_EXPANSION", cfg.LWR_EXPANSION))
                log_rank0("LWR residual scale init:", stage_get(cfg, stage, "LWR_RES_SCALE_INIT", cfg.LWR_RES_SCALE_INIT))
                log_rank0("LWR zero init final:", stage_get(cfg, stage, "LWR_ZERO_INIT_FINAL", cfg.LWR_ZERO_INIT_FINAL))
            log_rank0("Early stop:", {
                "patience": stage_get(cfg, stage, "EARLY_STOP_FLAT_PATIENCE", getattr(cfg, "EARLY_STOP_FLAT_PATIENCE", 0)),
                "min_delta": stage_get(cfg, stage, "EARLY_STOP_MIN_DELTA", getattr(cfg, "EARLY_STOP_MIN_DELTA", 0.0)),
            })
            log_rank0("Checkpoint retention:", {
                "top_k": getattr(cfg, "SAVE_TOP_K", 0),
                "recent_k": getattr(cfg, "SAVE_RECENT_K", 0),
            })

        if distributed and not is_xla:
            model = DDP(
                model,
                device_ids=[local_rank],
                output_device=local_rank,
                find_unused_parameters=False,
            )

        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=stage["LR"],
            weight_decay=stage["WEIGHT_DECAY"],
            fused=(device.type == "cuda")
        )

        total_steps = max(1, len(loader) * stage["EPOCHS"])
        warmup_steps = max(0, len(loader) * stage["WARMUP_EPOCHS"])
        scheduler = make_scheduler(
            optimizer,
            warmup_steps,
            total_steps,
            scheduler_type=stage.get("LR_SCHEDULER", cfg.LR_SCHEDULER),
            eta_min=stage.get("COSINE_ETA_MIN", cfg.COSINE_ETA_MIN),
        )
        scaler = torch.amp.GradScaler("cuda", enabled=cfg.PRECISION == "AMP" and device.type == "cuda")
        loss_fn = build_restoration_loss(cfg).to(device)
        if is_rank0():
            log_rank0(
                "Using LR scheduler:",
                stage.get("LR_SCHEDULER", cfg.LR_SCHEDULER),
                "warmup_steps=", warmup_steps,
                "total_steps=", total_steps,
                "eta_min=", stage.get("COSINE_ETA_MIN", cfg.COSINE_ETA_MIN),
            )
            log_rank0("Using restoration loss:", restoration_loss_description(cfg))
            log_rank0("Using pytorch_msssim:", HAS_PYTORCH_MSSSIM)
        metrics_path = out_dir / "metrics.csv"
        if is_rank0():
            with metrics_path.open("w", newline="") as f:
                writer = csv.writer(f)
                writer.writerow(["stage", "epoch", "train_total_loss", "train_l1", "train_ssim", "train_charbonnier", "train_gradient", "val_psnr", "val_rain", "val_snow", "lr", "elapsed_sec"])

        best = -1.0
        early_stop_best = -1.0
        flat_epochs = 0
        top = []
        recent = []
        for epoch in range(stage["EPOCHS"]):
            start = time.time()
            if sampler is not None:
                sampler.set_epoch(epoch)
            model.train()
            loss_keys = ["loss_total", "loss_l1", "loss_ssim", "loss_charbonnier", "loss_gradient"]
            running = {key: 0.0 for key in loss_keys}
            steps = 0
            pbar = tqdm(
                train_loader,
                total=len(loader),
                disable=(not is_rank0()) or (not cfg.SHOW_PROGRESS_BARS),
                dynamic_ncols=True,
                leave=False,
                desc=f"{stage_name} epoch {epoch}",
            ) if tqdm is not None else loader
            for meta, degraded, clean in pbar:
                degraded = degraded.to(device, non_blocking=True)
                clean = clean.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=cfg.PRECISION == "AMP" and device.type == "cuda"):
                    pred = model(degraded)
                    loss, parts = loss_fn(pred, clean)
                if not torch.isfinite(loss):
                    log_rank0("Skipping non-finite loss")
                    continue
                if device.type == "cuda":
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), stage["GRAD_CLIP"])
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), stage["GRAD_CLIP"])
                    if is_xla:
                        xm_mod, _, _ = require_xla()
                        xm_mod.optimizer_step(optimizer)
                        xm_mod.mark_step()
                    else:
                        optimizer.step()
                scheduler.step()
                steps += 1
                for key in running:
                    running[key] += float(parts[key].item())
                if is_rank0() and tqdm is not None:
                    pbar.set_postfix({
                        "loss": f"{running['loss_total'] / max(1, steps):.5f}",
                        "l1": f"{running['loss_l1'] / max(1, steps):.5f}",
                        "ssim": f"{running['loss_ssim'] / max(1, steps):.5f}",
                        "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
                    })
                if (
                    is_rank0()
                    and cfg.TRAIN_LOG_EVERY_N_STEPS
                    and (steps % cfg.TRAIN_LOG_EVERY_N_STEPS == 0)
                ):
                    log_rank0(
                        f"stage={stage_name} epoch={epoch} "
                        f"step={steps}/{len(loader)} "
                        f"train_total={running['loss_total'] / max(1, steps):.5f} "
                        f"l1={running['loss_l1'] / max(1, steps):.5f} "
                        f"ssim={running['loss_ssim'] / max(1, steps):.5f} "
                        f"charb={running['loss_charbonnier'] / max(1, steps):.5f} "
                        f"grad={running['loss_gradient'] / max(1, steps):.5f} "
                        f"lr={optimizer.param_groups[0]['lr']:.3e}"
                    )

            stats = torch.tensor(
                [running[key] for key in loss_keys] + [float(steps)],
                device=device,
                dtype=(torch.float32 if is_xla else torch.float64),
            )
            if is_xla:
                xm_mod, _, _ = require_xla()
                xm_mod.mark_step()
            elif distributed:
                dist.all_reduce(stats, op=dist.ReduceOp.SUM)
                dist.barrier()
            total_train_steps = max(1.0, float(stats[-1].item()))
            avg = {key: float(stats[idx].item() / total_train_steps) for idx, key in enumerate(loss_keys)}
            val = {"overall": float("nan"), "rain": float("nan"), "snow": float("nan")}
            stop_stage = False
            unwrapped = model.module if hasattr(model, "module") else model
            if is_rank0() and cfg.VALIDATE_EVERY_EPOCH:
                val = validate_model(unwrapped, val_set, device, use_tile=False, amp=cfg.PRECISION == "AMP", show_progress=cfg.SHOW_PROGRESS_BARS)
                avg_loss = avg["loss_total"]
                lr = optimizer.param_groups[0]["lr"]
                elapsed = time.time() - start
                ckpt_prefix = "lwr_" if stage_use_lwr else ""
                ckpt_name = f"{ckpt_prefix}epoch{epoch:03d}-psnr{val['overall']:.3f}.ckpt"
                ckpt_path = out_dir / ckpt_name
                save_checkpoint(ckpt_path, unwrapped, stage_name, epoch, val["overall"], cfg)
                if cfg.SAVE_LAST:
                    save_checkpoint(out_dir / "last.ckpt", unwrapped, stage_name, epoch, val["overall"], cfg)
                    save_full_checkpoint(
                        out_dir / "last_full.ckpt",
                        unwrapped,
                        optimizer,
                        scheduler,
                        scaler,
                        stage_name,
                        epoch,
                        val["overall"],
                        cfg,
                        global_step=(epoch + 1) * len(loader),
                    )
                top.append((val["overall"], ckpt_path))
                top = sorted(top, key=lambda x: x[0], reverse=True)
                save_top_k = max(0, int(getattr(cfg, "SAVE_TOP_K", 0)))
                top = top[:save_top_k] if save_top_k else []
                recent_k = max(0, int(getattr(cfg, "SAVE_RECENT_K", 0)))
                recent.append(ckpt_path)
                recent = recent[-recent_k:] if recent_k else []
                protected = {p.resolve() for _, p in top} | {p.resolve() for p in recent}
                for p in sorted(out_dir.glob("*epoch*-psnr*.ckpt")):
                    if p.resolve() not in protected and p.exists():
                        p.unlink()
                if val["overall"] > best:
                    best = val["overall"]
                    save_checkpoint(out_dir / "best.ckpt", unwrapped, stage_name, epoch, val["overall"], cfg)
                    previous_best = out_dir / "best.ckpt"
                patience = int(stage_get(cfg, stage, "EARLY_STOP_FLAT_PATIENCE", getattr(cfg, "EARLY_STOP_FLAT_PATIENCE", 0)) or 0)
                min_delta = float(stage_get(cfg, stage, "EARLY_STOP_MIN_DELTA", getattr(cfg, "EARLY_STOP_MIN_DELTA", 0.0)) or 0.0)
                if patience > 0:
                    if val["overall"] > early_stop_best + min_delta:
                        early_stop_best = val["overall"]
                        flat_epochs = 0
                    else:
                        flat_epochs += 1
                    if flat_epochs >= patience:
                        stop_stage = True
                        log_rank0(
                            f"Early stopping {stage_name}: val_psnr flat for {flat_epochs} epochs "
                            f"(best_for_stop={early_stop_best:.3f}, min_delta={min_delta:g})."
                        )
                with metrics_path.open("a", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow([stage_name, epoch, avg["loss_total"], avg["loss_l1"], avg["loss_ssim"], avg["loss_charbonnier"], avg["loss_gradient"], val["overall"], val["rain"], val["snow"], lr, elapsed])
                lwr_scale = ""
                if stage_use_lwr and hasattr(unwrapped, "local_weather_refine"):
                    lwr_scale = f" | lwr_scale={unwrapped.local_weather_refine.residual_scale.detach().float().mean().item():.4f}"
                log_rank0(
                    f"Epoch {epoch} | train_total={avg['loss_total']:.5f} | "
                    f"l1={avg['loss_l1']:.5f} | ssim={avg['loss_ssim']:.5f} | "
                    f"charb={avg['loss_charbonnier']:.5f} | grad={avg['loss_gradient']:.5f} | "
                    f"val_psnr={val['overall']:.3f} | rain={val['rain']:.3f} | "
                    f"snow={val['snow']:.3f} | lr={lr:.3e}{lwr_scale} | "
                    f"flat_epochs={flat_epochs} | time={elapsed:.1f}s"
                )
            stop_stage = sync_stop_flag(stop_stage, device, distributed=distributed, is_xla=is_xla)
            if is_xla:
                xm_mod, _, _ = require_xla()
                xm_mod.rendezvous(f"{stage_name}_epoch_{epoch}_done")
            elif distributed:
                dist.barrier()
            if stop_stage:
                break
        if is_rank0() and previous_best is None:
            previous_best = find_best_checkpoint(out_dir)

    if distributed and not is_xla:
        dist.destroy_process_group()


def resolve_inference_checkpoint(cfg):
    if cfg.INFERENCE_CKPT and cfg.INFERENCE_CKPT != "AUTO_BEST":
        return Path(cfg.INFERENCE_CKPT)
    inference_stage = normalize_run_only_stage(getattr(cfg, "INFERENCE_STAGE", None))
    run_only_stage = normalize_run_only_stage(cfg.RUN_ONLY_STAGE)
    if inference_stage is not None:
        candidates = [inference_stage]
    elif run_only_stage is not None:
        candidates = [run_only_stage]
    else:
        candidates = [s["NAME"] for s in cfg.STAGES][::-1]
    for name in candidates:
        ckpt = find_best_checkpoint(stage_dir(cfg, name))
        if ckpt is not None:
            return ckpt
        best = stage_dir(cfg, name) / "best.ckpt"
        if best.exists():
            return best
    raise FileNotFoundError("Could not resolve INFERENCE_CKPT=AUTO_BEST")


def tensor_to_uint8_chw(x):
    x = x.detach().float().cpu().squeeze(0).clamp(0, 1)
    arr = (x.numpy() * 255.0 + 0.5).clip(0, 255).astype(np.uint8)
    return arr



@torch.no_grad()
def run_lwr_smoke_test(cfg, device=None):
    if device is None:
        device = select_single_process_device(cfg)
    print("Running LWR smoke test on", device)

    base_model = PromptIR(
        decoder=True,
        dim=cfg.MODEL_DIM,
        num_blocks=cfg.NUM_BLOCKS,
        num_refinement_blocks=cfg.NUM_REFINEMENT_BLOCKS,
        prompt_len=cfg.PROMPT_LEN,
        heads=cfg.NUM_HEADS,
        ffn_expansion_factor=cfg.FFN_EXPANSION_FACTOR,
        bias=cfg.BIAS,
        LayerNorm_type=cfg.LAYER_NORM_TYPE,
        use_task_prompt_routing=False,
        num_tasks=2,
        use_local_weather_refine=False,
    ).to(device).eval()
    lwr_model = PromptIR(
        decoder=True,
        dim=cfg.MODEL_DIM,
        num_blocks=cfg.NUM_BLOCKS,
        num_refinement_blocks=cfg.NUM_REFINEMENT_BLOCKS,
        prompt_len=cfg.PROMPT_LEN,
        heads=cfg.NUM_HEADS,
        ffn_expansion_factor=cfg.FFN_EXPANSION_FACTOR,
        bias=cfg.BIAS,
        LayerNorm_type=cfg.LAYER_NORM_TYPE,
        use_task_prompt_routing=False,
        num_tasks=2,
        use_local_weather_refine=True,
        lwr_expansion=cfg.LWR_EXPANSION,
        lwr_res_scale_init=cfg.LWR_RES_SCALE_INIT,
        lwr_zero_init_final=cfg.LWR_ZERO_INIT_FINAL,
    ).to(device).eval()

    x = torch.randn(1, 3, 64, 64, device=device)
    base_y = base_model(x)
    lwr_y = lwr_model(x)
    assert base_y.shape == x.shape, f"Base output shape mismatch: {base_y.shape}"
    assert lwr_y.shape == x.shape, f"LWR output shape mismatch: {lwr_y.shape}"
    print("Smoke forward shapes OK:", tuple(base_y.shape), tuple(lwr_y.shape))

    init_ckpt = cfg.STAGES[0].get("INIT_CKPT") if cfg.STAGES else None
    if init_ckpt and init_ckpt != "AUTO_BEST_PREVIOUS" and Path(init_ckpt).exists():
        base_model = load_checkpoint(base_model, init_ckpt, map_location=device)
        lwr_model = load_checkpoint(
            lwr_model,
            init_ckpt,
            map_location=checkpoint_map_location_for_device(device),
            allow_partial_load_for_lwr=True,
        )
        base_y = base_model(x)
        lwr_y = lwr_model(x)
        max_abs_diff = (base_y - lwr_y).abs().max().item()
        print(f"LWR initial preservation max_abs_diff: {max_abs_diff:.8g}")
    else:
        print("Skipping checkpoint preservation check; init checkpoint not found locally:", init_ckpt)

    if "forward_with_padding" in globals():
        tta_y = forward_with_padding(lwr_model, x, use_tile=True, tile_size=64, tile_overlap=16)
        assert tta_y.shape == x.shape, f"Tile smoke shape mismatch: {tta_y.shape}"
        print("Tile path shape OK:", tuple(tta_y.shape))
    print("LWR smoke test: OK")


@torch.no_grad()
def run_final_validation(cfg, ckpt_path=None):
    root, split = build_or_load_split(cfg)
    device = select_single_process_device(cfg)
    ckpt = Path(ckpt_path) if ckpt_path else resolve_inference_checkpoint(cfg)
    inference_stage = stage_for_checkpoint(cfg, ckpt)
    model = build_model(cfg, inference_stage).to(device)
    load_checkpoint(
        model,
        ckpt,
        map_location=checkpoint_map_location_for_device(device),
        allow_partial_load_for_lwr=stage_uses_lwr(cfg, inference_stage) and stage_allows_lwr_partial_load(cfg, inference_stage),
    )
    print("Validating checkpoint:", ckpt)
    val_set = HW4RestorationDataset(root, split, "val", patch_size=128, augment=False)
    model_summary(model, cfg, device=device, stage=inference_stage)
    result = validate_model(
        model,
        val_set,
        device,
        use_tile=cfg.USE_TILED_VALIDATION,
        tile_size=cfg.TILE_SIZE,
        tile_overlap=cfg.TILE_OVERLAP,
        use_tta=cfg.USE_X8_TTA,
        amp=cfg.PRECISION == "AMP",
        show_progress=cfg.SHOW_PROGRESS_BARS,
    )
    print("Final validation:", result)
    return result, ckpt


@torch.no_grad()
def run_test_inference(cfg, ckpt_path=None):
    root = find_hw4_root(cfg.DATA_ROOT, cfg.AUTO_FIND_DATA_ROOT)
    test_dir = root / "test" / "degraded"
    device = select_single_process_device(cfg)
    ckpt = Path(ckpt_path) if ckpt_path else resolve_inference_checkpoint(cfg)
    inference_stage = stage_for_checkpoint(cfg, ckpt)
    model = build_model(cfg, inference_stage).to(device)
    load_checkpoint(
        model,
        ckpt,
        map_location=checkpoint_map_location_for_device(device),
        allow_partial_load_for_lwr=stage_uses_lwr(cfg, inference_stage) and stage_allows_lwr_partial_load(cfg, inference_stage),
    )
    model.eval()
    dataset = HW4TestDataset(test_dir)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    predictions = {}
    print("Inference checkpoint:", ckpt)
    for meta, degraded in loader:
        degraded = degraded.to(device)
        filename = meta["filename"][0]
        h = int(meta["height"].item())
        w = int(meta["width"].item())
        with torch.amp.autocast("cuda", enabled=cfg.PRECISION == "AMP" and device.type == "cuda"):
            pred = predict_with_tta(
                model,
                degraded,
                use_tile=cfg.USE_TILED_VALIDATION,
                tile_size=cfg.TILE_SIZE,
                tile_overlap=cfg.TILE_OVERLAP,
            ) if cfg.USE_X8_TTA else forward_with_padding(
                model,
                degraded,
                use_tile=cfg.USE_TILED_VALIDATION,
                tile_size=cfg.TILE_SIZE,
                tile_overlap=cfg.TILE_OVERLAP,
            )
        arr = tensor_to_uint8_chw(pred)
        if arr.shape != (3, h, w):
            raise RuntimeError(f"{filename}: expected {(3, h, w)}, got {arr.shape}")
        predictions[filename] = arr
    out_path = Path(cfg.SUBMISSION_NPZ_PATH)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez(out_path, **predictions)
    print(f"Saved {len(predictions)} predictions to {out_path}")
    return out_path


def validate_pred_npz(cfg):
    root = find_hw4_root(cfg.DATA_ROOT, cfg.AUTO_FIND_DATA_ROOT)
    test_dir = root / "test" / "degraded"
    expected = {p.name: Image.open(p).convert("RGB").size[::-1] for p in sorted_image_files(test_dir)}
    data = np.load(cfg.SUBMISSION_NPZ_PATH)
    keys = sorted(data.files, key=lambda x: int(Path(x).stem) if Path(x).stem.isdigit() else x)
    print("num_keys:", len(keys))
    print("first_keys:", keys[:5])
    assert set(keys) == set(expected.keys()), "pred.npz keys do not match test filenames"
    for key in keys:
        arr = data[key]
        h, w = expected[key]
        assert arr.shape == (3, h, w), f"{key}: expected {(3, h, w)}, got {arr.shape}"
        assert arr.dtype == np.uint8, f"{key}: expected uint8, got {arr.dtype}"
        assert arr.min() >= 0 and arr.max() <= 255, f"{key}: values out of uint8 range"
    print("pred.npz format: OK")
    return keys


def create_submission_zip(cfg):
    zip_path = Path(cfg.SUBMISSION_ZIP_PATH)
    npz_path = Path(cfg.SUBMISSION_NPZ_PATH)
    if not npz_path.exists():
        raise FileNotFoundError(npz_path)
    if zip_path.exists():
        zip_path.unlink()
    import zipfile
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(npz_path, arcname="pred.npz")
    print("Created:", zip_path)
    return zip_path
'''

TRAIN_DDP_CODE = r"""
import sys
sys.path.insert(0, "/kaggle/working")
import hw4_config as C
from hw4_lib import run_training

if __name__ == "__main__":
    run_training(C)
"""

INFER_CODE = r"""
import sys
sys.path.insert(0, "/kaggle/working")
import hw4_config as C
from hw4_lib import run_final_validation, run_test_inference, validate_pred_npz, create_submission_zip

if __name__ == "__main__":
    result, ckpt = run_final_validation(C)
    if C.MAKE_SUBMISSION:
        run_test_inference(C, ckpt)
        validate_pred_npz(C)
        create_submission_zip(C)
"""

Path("/kaggle/working/hw4_lib.py").write_text(LIB_CODE, encoding="utf-8")
Path("/kaggle/working/train_ddp.py").write_text(TRAIN_DDP_CODE, encoding="utf-8")
Path("/kaggle/working/infer_hw4.py").write_text(INFER_CODE, encoding="utf-8")
print("Wrote /kaggle/working/hw4_lib.py")
print("Wrote /kaggle/working/train_ddp.py")
print("Wrote /kaggle/working/infer_hw4.py")

sys.path.insert(0, "/kaggle/working")
import importlib
import hw4_config as C
import hw4_lib
importlib.reload(hw4_lib)

if hw4_lib.wants_tpu(C):
    device = "cpu"
    print("TPU selected; running notebook model summary on CPU to avoid pre-initializing XLA before spawned training.")
else:
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
available_stages = [stage["NAME"] for stage in C.STAGES]
if C.RUN_ONLY_STAGE and C.RUN_ONLY_STAGE in available_stages:
    summary_stage = hw4_lib.get_stage_by_name(C, C.RUN_ONLY_STAGE)
elif C.RUN_ONLY_STAGE:
    print(f"WARNING: RUN_ONLY_STAGE={C.RUN_ONLY_STAGE!r} is not in STAGES={available_stages}; using the first stage for summary only.")
    summary_stage = C.STAGES[0]
else:
    summary_stage = C.STAGES[0]
model = hw4_lib.build_model(C, summary_stage)
hw4_lib.model_summary(model, C, device=device, stage=summary_stage)

In [ ]:
write_config_py()
sys.path.insert(0, "/kaggle/working")
import importlib
import hw4_config as C
import hw4_lib
importlib.reload(hw4_lib)

loss_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
loss_fn = hw4_lib.build_restoration_loss(C).to(loss_device)
pred = torch.rand(1, 3, 128, 128, device=loss_device, requires_grad=True)
target = torch.rand(1, 3, 128, 128, device=loss_device)
total_loss, parts = loss_fn(pred, target)
print("Loss description:", hw4_lib.restoration_loss_description(C))
print("Using pytorch_msssim:", hw4_lib.HAS_PYTORCH_MSSSIM)
print("Sanity total loss:", float(total_loss.detach().cpu()))
print("Sanity components:", {key: float(value.detach().cpu()) for key, value in parts.items()})
total_loss.backward()
assert pred.grad is not None
assert torch.isfinite(pred.grad).all()
print("Loss backward sanity check: OK")
del loss_fn, pred, target, total_loss, parts
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
write_config_py()

accelerator = str(ACCELERATOR).strip().lower()
if accelerator == "auto":
    accelerator = "tpu" if (os.environ.get("PJRT_DEVICE", "").upper() == "TPU" or os.environ.get("TPU_NAME")) else "gpu"

if accelerator in {"tpu", "xla", "tpu-vm"}:
    # train_ddp.py will call xmp.spawn internally. Do not wrap TPU training in torchrun.
    cmd = [sys.executable, "/kaggle/working/train_ddp.py"]
else:
    gpu_count = torch.cuda.device_count()
    if NUM_GPUS == "AUTO":
        nproc = 2 if (USE_DDP and gpu_count >= 2) else min(1, gpu_count)
    else:
        nproc = int(NUM_GPUS)
        nproc = min(nproc, gpu_count)

    if nproc >= 2:
        cmd = [
            sys.executable, "-m", "torch.distributed.run",
            "--standalone",
            f"--nproc_per_node={nproc}",
            "/kaggle/working/train_ddp.py",
        ]
    else:
        cmd = [sys.executable, "/kaggle/working/train_ddp.py"]

print("Launching training with accelerator:", accelerator)
print(" ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(f"Training failed with return code {result.returncode}")

In [ ]:
write_config_py()
sys.path.insert(0, "/kaggle/working")
import importlib
import hw4_config as C
import hw4_lib
importlib.reload(hw4_lib)

if RUN_FINAL_VALIDATION:
    FINAL_VAL_RESULT, FINAL_CKPT = hw4_lib.run_final_validation(C)
    print("Compare against current best local tile+x8 validation: E031 = 29.8029, E030 public LB = 30.60")
    print("Selected checkpoint:", FINAL_CKPT)
else:
    FINAL_VAL_RESULT = None
    FINAL_CKPT = None
    print("RUN_FINAL_VALIDATION=False, skipping final tile/x8 validation.")
    print("Training checkpoints are saved under:", Path(OUTPUT_DIR) / "runs" / RUN_NAME)
    print("For a separate validation/submission run, set RUN_FINAL_VALIDATION=True or set INFERENCE_CKPT to a specific .ckpt path.")


In [ ]:
write_config_py()
if MAKE_SUBMISSION:
    sys.path.insert(0, "/kaggle/working")
    import importlib
    import hw4_config as C
    import hw4_lib
    importlib.reload(hw4_lib)
    PRED_NPZ_PATH = hw4_lib.run_test_inference(C, FINAL_CKPT if globals().get("FINAL_CKPT") is not None else None)
else:
    print("MAKE_SUBMISSION=False, skipping test inference.")

In [ ]:
if MAKE_SUBMISSION:
    sys.path.insert(0, "/kaggle/working")
    import hw4_config as C
    import hw4_lib
    SUBMISSION_KEYS = hw4_lib.validate_pred_npz(C)
else:
    print("MAKE_SUBMISSION=False, skipping pred.npz validation.")

In [ ]:
if MAKE_SUBMISSION:
    sys.path.insert(0, "/kaggle/working")
    import hw4_config as C
    import hw4_lib
    ZIP_PATH = hw4_lib.create_submission_zip(C)
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        print("Zip contents:", zf.namelist())
else:
    print("MAKE_SUBMISSION=False, skipping zip creation.")

In [ ]:
print("Done.")
print("If auto-detection fails, set DATA_ROOT to the folder containing train/degraded, train/clean, and test/degraded.")
print("224 FT: set RUN_ONLY_STAGE = FT224_STAGE_NAME and set BEST_192_CKPT_PATH, or let AUTO_BEST_192 search BEST_192_CKPT_GLOBS.")
print("LWR later: use a new Kaggle session, set RUN_ONLY_STAGE = LWR_STAGE_NAME, keep INIT_CKPT='AUTO_BEST_224', and set BEST_224_CKPT_PATH to the uploaded best 224 checkpoint.")
print("Flat validation early stop: EARLY_STOP_FLAT_PATIENCE =", EARLY_STOP_FLAT_PATIENCE, "min_delta=", EARLY_STOP_MIN_DELTA)
print("Checkpoint retention: top", SAVE_TOP_K, "+ recent", SAVE_RECENT_K, "for ensemble candidates.")
print("Post-train validation/submission defaults:", {"RUN_FINAL_VALIDATION": RUN_FINAL_VALIDATION, "MAKE_SUBMISSION": MAKE_SUBMISSION})
print("Resume from a checkpoint: set RESUME_CHECKPOINT = '/kaggle/working/output/.../last_full.ckpt' or another checkpoint.")
print("pred.npz:", SUBMISSION_NPZ_PATH)
print("submission.zip:", SUBMISSION_ZIP_PATH)
print("Current public-LB target to beat: E030 = 30.60")
